# PCOS Advanced RAG — Final Linear Layout-Aware Cleaning + Qdrant Cloud

This version uses one clear ingestion order: **PDF → layout-aware extraction → existing cleaning → generic repeated header/footer cleanup → broken/empty-page filtering → chunking → structural eligibility → retrieval**. Layout cleaning is completed before any chunking.


## Final Methodological Structure

- Docling layout-aware extraction is attempted first, with PyPDFLoader fallback.
- Existing project cleaning rules are preserved, including the current document-specific safe fixes.
- Generic repeated header/footer cleanup is applied **before chunking**.
- Broken/empty pages are excluded **before chunking**.
- Recursive and SLM-assisted chunking therefore operate on the same final cleaned pages.
- No scope classifier, no binary MiniLM gate, and no NLI model are used.
- MiniLM is used only for dense embeddings.
- The final SLM retrieval corpus is sent directly to Qdrant; there is no second corpus rebuild after chunking.


In [ ]:
import os
from qdrant_client import QdrantClient
from google.colab import userdata

# Retrieve Qdrant connection details (assuming they are defined later or need to be hardcoded/fetched)
QDRANT_URL = "https://74b795c6-af27-4520-b2fe-07363d53b8ea.us-east-1-1.aws.cloud.qdrant.io"
QDRANT_COLLECTION = "pcos_layout_cleaned_v1"

def get_qdrant_api_key():
    key = os.getenv("QDRANT_API_KEY")
    if key:
        return key.strip()
    try:
        key = userdata.get("QDRANT_API_KEY")
        if key:
            return str(key).strip()
    except Exception:
        pass
    return None

def connect_qdrant_cloud():
    api_key = get_qdrant_api_key()
    if not api_key:
        raise RuntimeError(
            "QDRANT_API_KEY is not set. In Colab, add it under "
            "Secrets with the name QDRANT_API_KEY."
        )
    client = QdrantClient(
        url=QDRANT_URL,
        api_key=api_key,
        timeout=60,
    )
    client.get_collections() # Test connection
    return client

try:
    client = connect_qdrant_cloud()
    print(f"Connected to Qdrant Cloud at {QDRANT_URL}")

    if client.collection_exists(collection_name=QDRANT_COLLECTION):
        client.delete_collection(collection_name=QDRANT_COLLECTION)
        print(f"Successfully deleted Qdrant collection: {QDRANT_COLLECTION}")
    else:
        print(f"Qdrant collection '{QDRANT_COLLECTION}' does not exist, no action needed.")

except RuntimeError as e:
    print(f"Qdrant operation skipped: {e}")
    print("Please ensure QDRANT_API_KEY is set in Colab Secrets or environment variables.")
except Exception as e:
    print(f"An unexpected error occurred during Qdrant operation: {e}")

Connected to Qdrant Cloud at https://74b795c6-af27-4520-b2fe-07363d53b8ea.us-east-1-1.aws.cloud.qdrant.io
Successfully deleted Qdrant collection: pcos_layout_cleaned_v1


## Mount Google Drive

### Code 001 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Define folders

### Code 002 — Define folders


In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/pcos_rag"

PDF_FOLDER = os.path.join(
    BASE_FOLDER,
    "documents"
)

CLEANED_FOLDER = os.path.join(
    BASE_FOLDER,
    "cleaned_documents"
)

CHUNKS_FOLDER = os.path.join(
    BASE_FOLDER,
    "chunks"
)

EVALUATION_FOLDER = os.path.join(
    BASE_FOLDER,
    "evaluation"
)

VECTOR_FOLDER = os.path.join(
    BASE_FOLDER,
    "vector_store"
)

for folder in [
    CLEANED_FOLDER,
    CHUNKS_FOLDER,
    EVALUATION_FOLDER,
    VECTOR_FOLDER
]:
    os.makedirs(
        folder,
        exist_ok=True
    )

print("PDF folder:", PDF_FOLDER)
print("Cleaned folder:", CLEANED_FOLDER)
print("Chunks folder:", CHUNKS_FOLDER)
print("Evaluation folder:", EVALUATION_FOLDER)
print("Vector folder:", VECTOR_FOLDER)

PDF folder: /content/drive/MyDrive/pcos_rag/documents
Cleaned folder: /content/drive/MyDrive/pcos_rag/cleaned_documents
Chunks folder: /content/drive/MyDrive/pcos_rag/chunks
Evaluation folder: /content/drive/MyDrive/pcos_rag/evaluation
Vector folder: /content/drive/MyDrive/pcos_rag/vector_store


Check PDF files

### Code 003 — Check PDF files


In [ ]:
if not os.path.isdir(PDF_FOLDER):
    raise FileNotFoundError(
        f"PDF folder not found: {PDF_FOLDER}\n"
        "Mount Google Drive first and make sure the documents folder exists."
    )

pdf_files = sorted([
    file
    for file in os.listdir(PDF_FOLDER)
    if file.lower().endswith(".pdf")
])

print("Total PDFs found:", len(pdf_files))

for index, file in enumerate(pdf_files, start=1):
    print(index, file)


Total PDFs found: 7
1 NICHDWHPCOSweb.pdf
2 PCOS-Gail-Busby-and-Montilla-002 (1).pdf
3 PCOS-Guideline-Summary-2023-2.pdf
4 Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf
5 Polycystic ovary syndrome.pdf
6 Updated-PMOS-Guideline-12-June26.pdf
7 What are the treatments for PCOS_ _ NICHD - Eunice Kennedy Shriver National Institute of Child Health and Human Development.pdf


Install extraction libraries

langchain-community	Provides LangChain integrations such as PDF loaders, vector-store integrations, document loaders, etc.
langchain-text-splitters	Splits large documents/text into smaller chunks for embeddings and RAG
pypdf	Reads and extracts text from PDF files

### Code 004 — pypdf	Reads and extracts text from PDF files


In [ ]:
!pip install -q \
langchain-community \
langchain-text-splitters \
pypdf \
docling \
qdrant-client


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.9/931.9 kB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 150.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.

### Code 005 — Imports and Setup


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

import pandas as pd
import re
import unicodedata

/tmp/ipykernel_4571/264822621.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Add source metadata

### Code 006 — Add source metadata


In [ ]:
SOURCE_METADATA = {

    "Polycystic ovary syndrome.pdf": {
        "organization": "World Health Organization",
        "source_type": "health_fact_sheet"
    },

    "What are the treatments for PCOS? | NICHD - Eunice Kennedy Shriver National Institute of Child Health and Human Development.pdf": {
        "organization": "NICHD",
        "source_type": "lifestyle_support"
    },

    "PCOS-Guideline-Summary-2023-2.pdf": {
        "organization": "International PCOS Guideline",
        "source_type": "guideline_summary"
    },

    "Updated-PMOS-Guideline-12-June26.pdf": {
        "organization": "International PCOS Guideline",
        "source_type": "clinical_guideline"
    },

    "NICHDWHPCOSweb.pdf": {
        "organization": "NICHD",
        "source_type": "patient_health_information"
    },

    "Polycystic Ovary Syndrome (PCOS) | ACOG.pdf": {
        "organization": (
            "American College of Obstetricians "
            "and Gynecologists"
        ),
        "source_type": "patient_guidance"
    },

    "PCOS-Gail-Busby-and-Montilla-002 (1).pdf": {
        "organization": (
            "Saint Mary's Managed Clinical Service, "
            "Manchester University NHS Foundation Trust"
        ),
        "source_type": "clinical_patient_guidance"
    }
}

## Global Layout-Aware Extraction and Cleaning Helpers — BEFORE Chunking

These helpers are defined before page extraction and chunking. Docling detects document layout and page furniture where possible; the existing three-stage cleaner is retained; a generic cross-page repeated header/footer safety net is also available.


In [ ]:
# ============================================================
# GLOBAL LAYOUT-AWARE PDF EXTRACTION + CLEANING
# Shared by BOTH the full-corpus rebuild and manual PDF input.
# ============================================================

import math
import os
import re
from collections import Counter
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

# Lazy Docling imports so the rest of the notebook can still be inspected
# even before the install cell has run.
_DOCLING_CONVERTER = None
_DOCLING_IMPORT_ERROR = None


def _get_docling_converter():
    global _DOCLING_CONVERTER, _DOCLING_IMPORT_ERROR

    if _DOCLING_CONVERTER is not None:
        return _DOCLING_CONVERTER

    try:
        from docling.datamodel.base_models import InputFormat
        from docling.datamodel.pipeline_options import PdfPipelineOptions
        from docling.document_converter import DocumentConverter, PdfFormatOption

        options = PdfPipelineOptions()
        options.do_ocr = True
        options.do_table_structure = True

        _DOCLING_CONVERTER = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=options
                )
            },
        )
        return _DOCLING_CONVERTER

    except Exception as exc:
        _DOCLING_IMPORT_ERROR = str(exc)
        return None


def _docling_keep_labels():
    """Keep body/evidence-bearing layout classes; omit page furniture/reference index."""
    try:
        from docling_core.types.doc import DocItemLabel
    except Exception:
        return None

    names = [
        "TEXT",
        "PARAGRAPH",
        "TITLE",
        "SECTION_HEADER",
        "LIST_ITEM",
        "TABLE",
        "CAPTION",
        "FOOTNOTE",
        "FORMULA",
    ]

    labels = {
        getattr(DocItemLabel, name)
        for name in names
        if hasattr(DocItemLabel, name)
    }
    return labels


def extract_pdf_pages_layout_aware(
    pdf_path,
    organization="unknown",
    source_type="unknown",
    manual_input=False,
):
    """
    Preferred extraction: Docling layout-aware PDF parsing.
    Fallback: PyPDFLoader.

    The Docling export intentionally includes only evidence-bearing labels,
    so detected PAGE_HEADER/PAGE_FOOTER furniture is not exported.
    """
    pdf_path = Path(pdf_path)
    filename = pdf_path.name
    converter = _get_docling_converter()

    if converter is not None:
        try:
            result = converter.convert(str(pdf_path))
            dl_doc = result.document
            keep_labels = _docling_keep_labels()

            page_numbers = sorted(
                int(page_no)
                for page_no in dl_doc.pages.keys()
            )

            pages = []
            for page_no in page_numbers:
                kwargs = {
                    "page_no": page_no,
                    "traverse_pictures": True,
                }
                if keep_labels:
                    kwargs["labels"] = keep_labels

                text = dl_doc.export_to_text(**kwargs)

                pages.append(
                    Document(
                        page_content=str(text or ""),
                        metadata={
                            "document_name": filename,
                            "pdf_page_number": int(page_no),
                            "organization": organization,
                            "source_type": source_type,
                            "manual_input": bool(manual_input),
                            "extraction_method": "docling_layout_aware",
                        },
                    )
                )

            if pages and any(p.page_content.strip() for p in pages):
                return pages, {
                    "parser": "docling_layout_aware",
                    "fallback_used": False,
                    "docling_error": None,
                }

        except Exception as exc:
            docling_error = str(exc)
        else:
            docling_error = "Docling returned no usable page text."
    else:
        docling_error = _DOCLING_IMPORT_ERROR or "Docling unavailable."

    # Fallback remains important for unusual PDFs or environments where
    # Docling models cannot initialize.
    loader = PyPDFLoader(str(pdf_path))
    raw_pages = loader.load()

    pages = []
    for page_index, page in enumerate(raw_pages, start=1):
        metadata = dict(page.metadata)
        metadata.update({
            "document_name": filename,
            "pdf_page_number": page_index,
            "organization": organization,
            "source_type": source_type,
            "manual_input": bool(manual_input),
            "extraction_method": "pypdf_fallback",
        })
        pages.append(
            Document(
                page_content=page.page_content,
                metadata=metadata,
            )
        )

    return pages, {
        "parser": "pypdf_fallback",
        "fallback_used": True,
        "docling_error": docling_error,
    }


def _normalize_edge_line(line):
    """Normalize an edge line so page-number variants can still match."""
    line = str(line or "").strip().lower()
    line = re.sub(r"\b\d+\b", "<num>", line)
    line = re.sub(r"\s+", " ", line)
    line = re.sub(r"[^a-z0-9<>%&/()\- .:]", "", line)
    return line.strip()


def remove_repeated_edge_lines(
    page_texts,
    edge_line_count=4,
    min_page_fraction=0.60,
    min_pages=3,
):
    """
    Generic repeated header/footer cleanup across pages.

    Only the first/last few non-empty lines are considered, and removal is
    enabled only for documents with at least `min_pages` pages. This makes the
    rule conservative and avoids stripping normal body text from short PDFs.
    """
    page_texts = [str(t or "") for t in page_texts]
    n_pages = len(page_texts)

    if n_pages < min_pages:
        return page_texts, {
            "repeated_edge_patterns": [],
            "removed_line_count": 0,
        }

    per_page_edge_keys = []
    for text in page_texts:
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        edge_lines = lines[:edge_line_count] + lines[-edge_line_count:]
        keys = {
            _normalize_edge_line(line)
            for line in edge_lines
            if len(_normalize_edge_line(line)) >= 3
        }
        per_page_edge_keys.append(keys)

    counts = Counter()
    for keys in per_page_edge_keys:
        counts.update(keys)

    threshold = max(
        min_pages,
        int(math.ceil(n_pages * min_page_fraction)),
    )

    repeated = {
        key
        for key, count in counts.items()
        if count >= threshold
    }

    cleaned_pages = []
    removed_count = 0

    for text in page_texts:
        raw_lines = text.splitlines()
        nonempty_positions = [
            idx for idx, line in enumerate(raw_lines)
            if line.strip()
        ]
        edge_positions = set(
            nonempty_positions[:edge_line_count]
            + nonempty_positions[-edge_line_count:]
        )

        kept = []
        for idx, line in enumerate(raw_lines):
            key = _normalize_edge_line(line)
            if idx in edge_positions and key in repeated:
                removed_count += 1
                continue
            kept.append(line)

        cleaned_pages.append("\n".join(kept).strip())

    return cleaned_pages, {
        "repeated_edge_patterns": sorted(repeated),
        "removed_line_count": int(removed_count),
    }


def is_broken_spaced_text(text):
    """Generic safety check for severely broken letter-spaced extraction."""
    words = str(text or "").split()
    if len(words) < 20:
        return False

    single_chars = sum(
        1
        for word in words
        if len(word) == 1 and word.isalpha()
    )
    return (single_chars / len(words)) >= 0.30


def clean_extracted_pdf_pages(raw_pages):
    """
    Apply the exact same cleaning sequence to every PDF:
      1) existing clean_text
      2) existing second_pass_clean (keeps current filename-specific fixes)
      3) existing final_safe_clean
      4) generic cross-page repeated header/footer cleanup
      5) empty/broken-page checks
    """
    stage3_docs = []

    for page in raw_pages:
        filename = page.metadata.get("document_name")
        page_number = page.metadata.get("pdf_page_number")

        stage_1 = clean_text(page.page_content)
        stage_2 = second_pass_clean(
            stage_1,
            document_name=filename,
        )
        stage_3 = final_safe_clean(
            stage_2,
            document_name=filename,
            page_number=page_number,
        )

        stage3_docs.append(
            Document(
                page_content=stage_3,
                metadata=page.metadata.copy(),
            )
        )

    repeated_cleaned_texts, repeated_report = remove_repeated_edge_lines(
        [doc.page_content for doc in stage3_docs]
    )

    cleaned_pages = []
    excluded_pages = []

    for doc, final_text in zip(stage3_docs, repeated_cleaned_texts):
        filename = doc.metadata.get("document_name")
        page_number = doc.metadata.get("pdf_page_number")

        known_broken = False
        if "BROKEN_SPACED_EXCLUSIONS" in globals():
            known_broken = (
                filename in BROKEN_SPACED_EXCLUSIONS
                and page_number in BROKEN_SPACED_EXCLUSIONS[filename]
            )

        if not str(final_text).strip():
            excluded_pages.append({
                "document": filename,
                "page": page_number,
                "reason": "empty_page",
            })
            continue

        if known_broken or is_broken_spaced_text(final_text):
            excluded_pages.append({
                "document": filename,
                "page": page_number,
                "reason": "broken_spaced_text",
            })
            continue

        metadata = doc.metadata.copy()
        metadata["repeated_edge_cleanup_applied"] = True

        cleaned_pages.append(
            Document(
                page_content=str(final_text).strip(),
                metadata=metadata,
            )
        )

    return cleaned_pages, excluded_pages, repeated_report


def load_and_clean_pdf(
    pdf_path,
    organization="unknown",
    source_type="unknown",
    manual_input=False,
):
    """One shared entry point for every new PDF in this project."""
    raw_pages, extraction_report = extract_pdf_pages_layout_aware(
        pdf_path=pdf_path,
        organization=organization,
        source_type=source_type,
        manual_input=manual_input,
    )

    cleaned_pages, excluded_pages, repeated_report = clean_extracted_pdf_pages(
        raw_pages
    )

    return {
        "raw_pages": raw_pages,
        "cleaned_pages": cleaned_pages,
        "excluded_pages": excluded_pages,
        "extraction_report": extraction_report,
        "repeated_edge_report": repeated_report,
    }


print("✅ Global layout-aware PDF cleaner ready.")
print("Preferred parser: Docling | fallback: PyPDFLoader")
print("Existing hard-coded cleaning rules: preserved")

def stable_chunk_id(
    document_name,
    page_number,
    page_chunk_index,
    chunk_text,
):
    """Stable UUID5 for manually/newly ingested chunks."""
    import uuid
    seed = (
        f"{document_name}|{page_number}|"
        f"{page_chunk_index}|{chunk_text}"
    )
    return str(uuid.uuid5(uuid.NAMESPACE_URL, seed))



✅ Global layout-aware PDF cleaner ready.
Preferred parser: Docling | fallback: PyPDFLoader
Existing hard-coded cleaning rules: preserved


Extract all PDF pages with layout-aware parsing

Docling is attempted first so page headers/footers and document layout are handled before the text-cleaning and chunking stages. PyPDFLoader is used only as a fallback.


### Code 007 — Extract all PDF pages


In [ ]:
raw_documents = []
loading_errors = []
extraction_reports = []

for filename in pdf_files:

    file_path = os.path.join(
        PDF_FOLDER,
        filename
    )

    print(
        "\nLoading:",
        filename
    )

    source_metadata = SOURCE_METADATA.get(
        filename,
        {
            "organization": "unknown",
            "source_type": "unknown"
        }
    )

    try:
        pages, extraction_report = extract_pdf_pages_layout_aware(
            pdf_path=file_path,
            organization=source_metadata["organization"],
            source_type=source_metadata["source_type"],
            manual_input=False,
        )

        raw_documents.extend(pages)

        extraction_reports.append({
            "document": filename,
            "pages": len(pages),
            "parser": extraction_report.get("parser"),
            "fallback_used": extraction_report.get("fallback_used"),
            "docling_error": extraction_report.get("docling_error"),
        })

        print(
            "Pages:",
            len(pages),
            "| Parser:",
            extraction_report.get("parser")
        )

    except Exception as error:
        loading_errors.append({
            "document": filename,
            "error": str(error)
        })


print(
    "\nTOTAL EXTRACTED PAGES:",
    len(raw_documents)
)

print(
    "LOADING ERRORS:",
    len(loading_errors)
)

if extraction_reports:
    display(pd.DataFrame(extraction_reports))

if loading_errors:
    display(pd.DataFrame(loading_errors))



Loading: NICHDWHPCOSweb.pdf
Pages: 2 | Parser: docling_layout_aware

Loading: PCOS-Gail-Busby-and-Montilla-002 (1).pdf
Pages: 6 | Parser: docling_layout_aware

Loading: PCOS-Guideline-Summary-2023-2.pdf


Pages: 35 | Parser: docling_layout_aware

Loading: Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf
Pages: 5 | Parser: docling_layout_aware

Loading: Polycystic ovary syndrome.pdf
Pages: 7 | Parser: docling_layout_aware

Loading: Updated-PMOS-Guideline-12-June26.pdf


Pages: 274 | Parser: docling_layout_aware

Loading: What are the treatments for PCOS_ _ NICHD - Eunice Kennedy Shriver National Institute of Child Health and Human Development.pdf
Pages: 3 | Parser: docling_layout_aware

TOTAL EXTRACTED PAGES: 332
LOADING ERRORS: 0


,document,pages,parser,fallback_used,docling_error
0,NICHDWHPCOSweb.pdf,2,docling_layout_aware,False,None
1,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,6,docling_layout_aware,False,None
2,PCOS-Guideline-Summary-2023-2.pdf,35,docling_layout_aware,False,None
3,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,5,docling_layout_aware,False,None
4,Polycystic ovary syndrome.pdf,7,docling_layout_aware,False,None
5,Updated-PMOS-Guideline-12-June26.pdf,274,docling_layout_aware,False,None
6,What are the treatments for PCOS_ _ NICHD - Eu...,3,docling_layout_aware,False,None


Preview raw extraction


### Code 008 — Preview raw extraction


In [ ]:
for index, doc in enumerate(
    raw_documents[:20],
    start=1
):

    print(
        "\n--- RAW PAGE",
        index,
        "---"
    )

    print(
        "Document:",
        doc.metadata.get(
            "document_name"
        )
    )

    print(
        "Page:",
        doc.metadata.get(
            "pdf_page_number"
        )
    )

    print(
        "Source:",
        doc.metadata.get(
            "organization"
        )
    )

    print()

    print(
        doc.page_content[:1000]
    )

    print(
        "=" * 100
    )


--- RAW PAGE 1 ---
Document: NICHDWHPCOSweb.pdf
Page: 1
Source: NICHD

Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS describes a group of symptoms related to a hormonal imbalance that can affect women and girls of reproductive age. A PCOS diagnosis means at least two of the following features are present: 1) sporadic ovulation, resulting in irregular or no menstrual periods; 2) high levels of androgen hormones, or signs of high androgen levels, such as acne that does not respond well to usual treatments; 3) growths in one or both ovaries, often clumps of ovarian follicles that have stopped developing. People with PCOS can also be at higher risk for infertility, diabetes and metabolic conditions, and cardiovascular disease. They may also have unwanted hair growth, weight gain, dark skin patches, acne, and heavy menst

Create the first cleaning function

### Code 009 — Create the first cleaning function


In [ ]:
def normalize_unicode(text):

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    replacements = {

        "\u00a0": " ",

        "\u200b": "",

        "\ufeff": "",

        "ﬁ": "fi",

        "ﬂ": "fl",

        "–": "-",

        "—": "-",

        "’": "'",

        "“": '"',

        "”": '"'
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new
        )

    return text

Now webpage noise cleaning:

### Code 010 — Now webpage noise cleaning:


In [ ]:
def remove_web_noise(text):

    patterns = [

        r"https?://\S+",

        r"www\.\S+",

        r"\bPage\s+\d+\s+of\s+\d+\b",

        r"^\s*\d{4}-\d{2}-\d{2},\s*\d{1,2}:\d{2}",

        r"\butm[\s_%-]*source\s*=\s*\S+",

        r"\bDonate\b",

        r"\bPrintShare\b",

        r"\bPrint\b",

        r"\bShare\b",

        r"\bReading time:\s*\d+\s*min(?:\s*\(\d+\s*words\))?"
    ]

    for pattern in patterns:

        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE |
                  re.MULTILINE
        )

    return text

Whitespace cleaning:

### Code 011 — Whitespace cleaning:


In [ ]:
def normalize_whitespace(text):

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r" *\n *",
        "\n",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    text = re.sub(
        r"\s+([,.;:!?])",
        r"\1",
        text
    )

    return text.strip()

Combine all:

### Code 012 — Combine all:


In [ ]:
def clean_text(text):

    if not isinstance(
        text,
        str
    ):
        return ""

    text = normalize_unicode(
        text
    )

    text = remove_web_noise(
        text
    )

    text = normalize_whitespace(
        text
    )

    return text

Apply cleaning

### Code 013 — Apply cleaning


In [ ]:
cleaned_documents = []

for doc in raw_documents:

    cleaned_text = clean_text(
        doc.page_content
    )

    cleaned_doc = Document(

        page_content=
            cleaned_text,

        metadata=
            doc.metadata.copy()
    )

    cleaned_documents.append(
        cleaned_doc
    )


print(
    "Cleaned pages:",
    len(cleaned_documents)
)

Cleaned pages: 332


Preview before vs after

### Code 014 — Preview before vs after


In [ ]:
for index in range(
    min(
        10,
        len(raw_documents)
    )
):

    raw_doc = (
        raw_documents[index]
    )

    clean_doc = (
        cleaned_documents[index]
    )

    print(
        "\nDOCUMENT:",
        raw_doc.metadata.get(
            "document_name"
        )
    )

    print(
        "PAGE:",
        raw_doc.metadata.get(
            "pdf_page_number"
        )
    )

    print(
        "\n--- BEFORE ---"
    )

    print(
        raw_doc.page_content[:700]
    )

    print(
        "\n--- AFTER ---"
    )

    print(
        clean_doc.page_content[:700]
    )

    print(
        "=" * 100
    )


DOCUMENT: NICHDWHPCOSweb.pdf
PAGE: 1

--- BEFORE ---
Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS describes a group of symptoms related to a hormonal imbalance that can affect women and girls of reproductive age. A PCOS diagnosis means at least two of the following features are present: 1) sporadic ovulation, resulting in irregular or no menstrual periods; 2) high levels of androgen hormones, or signs of high androgen levels, such as acne that does not respond well to usual treatments; 3) growths in one or both ovaries, often clumps of ovarian follicles that have stopped de

--- AFTER ---
Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS describes a 

I suggest adding a second-pass cleaner that only fixes safe structural noise:

### Code 015 — I suggest adding a second-pass cleaner that only fixes safe structural noise:


In [ ]:
def second_pass_clean(text, document_name=None):

    if not text:
        return ""

    # -----------------------------------
    # 1. Fix known joined header
    # -----------------------------------
    text = text.replace(
        "Manchester University NHS Foundation TrustPathway Guide",
        "Manchester University NHS Foundation Trust\nPathway Guide"
    )

    # -----------------------------------
    # 2. Remove standalone page numbers
    # Example:
    # 2
    # Abstract
    # -----------------------------------
    text = re.sub(
        r"(?m)^\s*\d{1,3}\s*$",
        "",
        text
    )

    # -----------------------------------
    # 3. Remove repeated NHS page header
    # But keep it if it is the first meaningful line
    # -----------------------------------
    if document_name == "PCOS-Gail-Busby-and-Montilla-002 (1).pdf":

        repeated_header = (
            r"(?m)^Saint Mary's Managed Clinical Service\s*\n"
            r"Manchester University NHS Foundation Trust\s*\n?"
        )

        matches = list(
            re.finditer(
                repeated_header,
                text
            )
        )

        if len(matches) > 0:
            # remove the repeated header from page text
            text = re.sub(
                repeated_header,
                "",
                text
            )

    # -----------------------------------
    # 4. Remove repeated guideline header
    # -----------------------------------
    if document_name in [
        "PCOS-Guideline-Summary-2023-2.pdf",
        "Updated-PMOS-Guideline-12-June26.pdf"
    ]:

        text = re.sub(
            r"(?im)^International evidence-based guideline.*?(?:Summary)?\s*$",
            "",
            text
        )

    # -----------------------------------
    # 5. Fix excessive blank lines again
    # -----------------------------------
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

Then apply it:

### Code 016 — Then apply it:


In [ ]:
final_cleaned_documents = []

for doc in cleaned_documents:

    filename = doc.metadata.get(
        "document_name"
    )

    final_text = second_pass_clean(
        doc.page_content,
        document_name=filename
    )

    final_doc = Document(
        page_content=final_text,
        metadata=doc.metadata.copy()
    )

    final_cleaned_documents.append(
        final_doc
    )

print(
    "Final cleaned pages:",
    len(final_cleaned_documents)
)

Final cleaned pages: 332


Then compare again:

### Code 017 — Then compare again:


In [ ]:
for index in range(
    min(15, len(cleaned_documents))
):

    before = cleaned_documents[index]
    after = final_cleaned_documents[index]

    print(
        "\nDOCUMENT:",
        before.metadata.get("document_name")
    )

    print(
        "PAGE:",
        before.metadata.get("pdf_page_number")
    )

    print("\n--- FIRST PASS ---")
    print(before.page_content[:700])

    print("\n--- SECOND PASS ---")
    print(after.page_content[:700])

    print("=" * 100)


DOCUMENT: NICHDWHPCOSweb.pdf
PAGE: 1

--- FIRST PASS ---
Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS describes a group of symptoms related to a hormonal imbalance that can affect women and girls of reproductive age. A PCOS diagnosis means at least two of the following features are present: 1) sporadic ovulation, resulting in irregular or no menstrual periods; 2) high levels of androgen hormones, or signs of high androgen levels, such as acne that does not respond well to usual treatments; 3) growths in one or both ovaries, often clumps of ovarian follicles that have stopped de

--- SECOND PASS ---
Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS de

We can now save this second-pass version

Use final_cleaned_documents, not the earlier cleaned_documents.

### Code 018 — We can now save this second-pass version


In [ ]:
from collections import defaultdict
import os

final_docs_by_file = defaultdict(list)

for doc in final_cleaned_documents:
    filename = doc.metadata.get("document_name")
    final_docs_by_file[filename].append(doc)


saved_final_files = []

for filename, pages in final_docs_by_file.items():

    pages = sorted(
        pages,
        key=lambda x: x.metadata.get(
            "pdf_page_number",
            0
        )
    )

    clean_filename = (
        os.path.splitext(filename)[0]
        + "_FINAL_CLEANED.txt"
    )

    output_path = os.path.join(
        CLEANED_FOLDER,
        clean_filename
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for page in pages:

            page_number = page.metadata.get(
                "pdf_page_number"
            )

            f.write(
                f"\n\n===== PDF PAGE {page_number} =====\n\n"
            )

            f.write(page.page_content)

    saved_final_files.append(
        output_path
    )

print(
    "FINAL CLEANED DOCUMENTS SAVED:",
    len(saved_final_files)
)

for path in saved_final_files:
    print(path)

FINAL CLEANED DOCUMENTS SAVED: 7
/content/drive/MyDrive/pcos_rag/cleaned_documents/NICHDWHPCOSweb_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/PCOS-Gail-Busby-and-Montilla-002 (1)_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/PCOS-Guideline-Summary-2023-2_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/Polycystic Ovary Syndrome (PCOS) _ ACOG_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/Polycystic ovary syndrome_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/Updated-PMOS-Guideline-12-June26_FINAL_CLEANED.txt
/content/drive/MyDrive/pcos_rag/cleaned_documents/What are the treatments for PCOS_ _ NICHD - Eunice Kennedy Shriver National Institute of Child Health and Human Development_FINAL_CLEANED.txt


### Code 019 — URL still remaining


In [ ]:
import re
import pandas as pd

cleaning_audit = []

for doc in final_cleaned_documents:

    text = doc.page_content

    problems = []

    # URL still remaining
    if re.search(
        r"https?://|www\.",
        text,
        flags=re.I
    ):
        problems.append("URL_REMAINS")

    # Replacement / encoding character
    if "�" in text:
        problems.append("ENCODING_CHARACTER")

    # Suspicious long spaced text
    tokens = text.split()

    if len(tokens) >= 20:

        single_letter_tokens = sum(
            1
            for token in tokens
            if len(token) == 1
            and token.isalpha()
        )

        ratio = (
            single_letter_tokens
            / len(tokens)
        )

        if ratio >= 0.30:
            problems.append(
                "BROKEN_SPACED_TEXT"
            )

    # Very short page
    if len(tokens) < 15:
        problems.append("VERY_SHORT_PAGE")

    # Multiple spaces
    if re.search(r" {3,}", text):
        problems.append(
            "EXCESSIVE_SPACES"
        )

    # Possible repeated page header
    if re.search(
        r"International evidence-based guideline "
        r"for the assessment and management",
        text,
        flags=re.I
    ):
        problems.append(
            "GUIDELINE_HEADER_REMAINS"
        )

    # Suspicious isolated page number
    if re.search(
        r"(?m)^\s*\d{1,3}\s*$",
        text
    ):
        problems.append(
            "PAGE_NUMBER_REMAINS"
        )

    if problems:

        cleaning_audit.append({
            "document":
                doc.metadata.get(
                    "document_name"
                ),
            "page":
                doc.metadata.get(
                    "pdf_page_number"
                ),
            "problems":
                ", ".join(problems),
            "preview":
                text[:300]
        })


cleaning_audit_df = pd.DataFrame(
    cleaning_audit
)

print(
    "Pages flagged:",
    len(cleaning_audit_df)
)

display(
    cleaning_audit_df
)

Pages flagged: 23


,document,page,problems,preview
0,Updated-PMOS-Guideline-12-June26.pdf,6,GUIDELINE_HEADER_REMAINS,Disclaimer\n\nThe Centre for Research Excellen...
1,Updated-PMOS-Guideline-12-June26.pdf,9,VERY_SHORT_PAGE,Chapter Three\n\nChapter Six
2,Updated-PMOS-Guideline-12-June26.pdf,10,GUIDELINE_HEADER_REMAINS,Preface\n\nThis International Evidence-based G...
3,Updated-PMOS-Guideline-12-June26.pdf,15,GUIDELINE_HEADER_REMAINS,Governance included international representati...
4,Updated-PMOS-Guideline-12-June26.pdf,101,VERY_SHORT_PAGE,Chapter Three Lifestyle management\n\nO\n\nS\n\nD
5,Updated-PMOS-Guideline-12-June26.pdf,118,VERY_SHORT_PAGE,Chapter Four\n\nManagement of non-fertility fe...
6,Updated-PMOS-Guideline-12-June26.pdf,145,VERY_SHORT_PAGE,Chapter Five Assessment and treatment of infer...
7,Updated-PMOS-Guideline-12-June26.pdf,174,VERY_SHORT_PAGE,Chapter Six Guideline development methods\n\nO...
8,Updated-PMOS-Guideline-12-June26.pdf,175,GUIDELINE_HEADER_REMAINS,This guideline was developed as outlined in Na...
9,Updated-PMOS-Guideline-12-June26.pdf,179,GUIDELINE_HEADER_REMAINS,The clinical questions addressed by each GDG a...


### Code 020 — Export Results


In [ ]:
cleaning_audit_df.to_csv(
    os.path.join(
        EVALUATION_FOLDER,
        "final_cleaning_audit.csv"
    ),
    index=False
)

Then do one more very useful check

We shouldn't inspect only the first 15 pages because the big updated guideline is where the broken-spacing problem previously appeared.

This code checks the first, middle, and last page of every PDF:

### Code 021 — Then do one more very useful check


In [ ]:
for filename, pages in final_docs_by_file.items():

    pages = sorted(
        pages,
        key=lambda x:
            x.metadata.get(
                "pdf_page_number",
                0
            )
    )

    if not pages:
        continue

    indexes = {
        0,
        len(pages) // 2,
        len(pages) - 1
    }

    print(
        "\n\n",
        "#" * 120
    )

    print(
        "DOCUMENT:",
        filename
    )

    for idx in sorted(indexes):

        doc = pages[idx]

        print(
            "\n--- PDF PAGE",
            doc.metadata.get(
                "pdf_page_number"
            ),
            "---"
        )

        print(
            doc.page_content[:1500]
        )



 ########################################################################################################################
DOCUMENT: NICHDWHPCOSweb.pdf

--- PDF PAGE 1 ---
Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

PCOS describes a group of symptoms related to a hormonal imbalance that can affect women and girls of reproductive age. A PCOS diagnosis means at least two of the following features are present: 1) sporadic ovulation, resulting in irregular or no menstrual periods; 2) high levels of androgen hormones, or signs of high androgen levels, such as acne that does not respond well to usual treatments; 3) growths in one or both ovaries, often clumps of ovarian follicles that have stopped developing. People with PCOS can also be at higher risk for infertility, diabetes and metabolic conditions, and cardiovascular di

So I would now create a third and final cleaning pass with only safe rules:

### Code 022 — So I would now create a third and final cleaning pass with only safe rules:


In [ ]:
def final_safe_clean(text, document_name=None, page_number=None):

    if not text:
        return ""

    # 1. Remove browser-export tracking noise
    text = re.sub(
        r"(?im)^_source=chatgpt\.com\s*$",
        "",
        text
    )

    # 2. Remove leftover URLs
    text = re.sub(
        r"https?://\S+",
        " ",
        text
    )

    # 3. Remove common browser footer/navigation fragments
    browser_noise_patterns = [
        r"DictionaryAbout ACOGSupporters",
        r"acog\.orgACOG ClinicalFor PatientsStore",
        r"HealthyLiving HealthyTeens HealthyAging PregnancyTopicsA-Z Experts &Stories",
        r"En Español",
        r"Related A-Z Topics",
        r"NICHD News and Features"
    ]

    for pattern in browser_noise_patterns:
        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.I
        )

    # 4. Remove repeated exact guideline running header
    text = re.sub(
        r"(?im)^International evidence-based guideline for the assessment and management of polycystic ovary syndrome 2023 - Summary\s*$",
        "",
        text
    )

    # 5. Remove standalone page markers like 132 or 95 
    text = re.sub(
        r"(?m)^\s*\d+\s*[]?\s*$",
        "",
        text
    )

    # 6. Normalize repeated whitespace
    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

### Code 023 — Define final_v2_documents


In [ ]:
final_v2_documents = []

for doc in final_cleaned_documents:

    filename = doc.metadata.get(
        "document_name"
    )

    page_number = doc.metadata.get(
        "pdf_page_number"
    )

    text = final_safe_clean(
        doc.page_content,
        document_name=filename,
        page_number=page_number
    )

    new_doc = Document(
        page_content=text,
        metadata=doc.metadata.copy()
    )

    final_v2_documents.append(
        new_doc
    )

print(
    "Final V2 pages:",
    len(final_v2_documents)
)

Final V2 pages: 332


### Global repeated header/footer cleanup — before chunking

After the existing three cleaning passes, repeated top/bottom lines are detected across pages of each document and removed conservatively. This is the last text-cleaning step before pages are admitted to chunking.


In [ ]:
from collections import defaultdict

_docs_by_file_for_edge_cleaning = defaultdict(list)
for doc in final_v2_documents:
    _docs_by_file_for_edge_cleaning[
        doc.metadata.get("document_name", "unknown")
    ].append(doc)

layout_cleaned_documents = []
repeated_edge_cleanup_reports = []

for filename, docs in _docs_by_file_for_edge_cleaning.items():
    docs = sorted(
        docs,
        key=lambda d: d.metadata.get("pdf_page_number", 0)
    )

    cleaned_texts, edge_report = remove_repeated_edge_lines(
        [doc.page_content for doc in docs]
    )

    repeated_edge_cleanup_reports.append({
        "document": filename,
        "page_count": len(docs),
        "repeated_edge_patterns": len(
            edge_report.get("repeated_edge_patterns", [])
        ),
        "removed_line_count": edge_report.get(
            "removed_line_count", 0
        ),
    })

    for doc, cleaned_text in zip(docs, cleaned_texts):
        layout_cleaned_documents.append(
            Document(
                page_content=cleaned_text,
                metadata=doc.metadata.copy()
            )
        )

# From this point onward, the notebook uses ONLY the fully cleaned pages.
final_v2_documents = layout_cleaned_documents

print(
    "Pages after global repeated-edge cleanup:",
    len(final_v2_documents)
)

if repeated_edge_cleanup_reports:
    display(pd.DataFrame(repeated_edge_cleanup_reports))


Pages after global repeated-edge cleanup: 332


,document,page_count,repeated_edge_patterns,removed_line_count
0,NICHDWHPCOSweb.pdf,2,0,0
1,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,6,0,0
2,PCOS-Guideline-Summary-2023-2.pdf,35,1,26
3,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,5,0,0
4,Polycystic ovary syndrome.pdf,7,0,0
5,Updated-PMOS-Guideline-12-June26.pdf,274,0,0
6,What are the treatments for PCOS_ _ NICHD - Eu...,3,0,0


Then explicitly flag the broken-spaced pages so they do not enter chunking:

### Code 024 — Then explicitly flag the broken-spaced pages so they do not enter chunking:


In [ ]:
BROKEN_SPACED_EXCLUSIONS = {
    "Updated-PMOS-Guideline-12-June26.pdf": {
        2, 3, 4, 5
    }
}

And when preparing the chunkable corpus:

this code cleans/filter-checks your PDF pages and keeps only usable pages for the next chunking step.

### Code 025 — And when preparing the chunkable corpus:


In [ ]:
chunkable_documents = []
excluded_for_chunking = []

for doc in final_v2_documents:

    filename = doc.metadata.get(
        "document_name"
    )

    page_number = doc.metadata.get(
        "pdf_page_number"
    )

    text = str(doc.page_content or "").strip()
    words = text.split()

    known_broken = (
        filename in BROKEN_SPACED_EXCLUSIONS
        and page_number in BROKEN_SPACED_EXCLUSIONS[filename]
    )

    # Exclude known or generically detected broken-spaced pages.
    if known_broken or is_broken_spaced_text(text):
        excluded_for_chunking.append({
            "document": filename,
            "page": page_number,
            "reason": "broken_spaced_text"
        })
        continue

    # Exclude empty pages.
    if len(words) == 0:
        excluded_for_chunking.append({
            "document": filename,
            "page": page_number,
            "reason": "empty_page"
        })
        continue

    chunkable_documents.append(doc)

print("Chunkable pages:", len(chunkable_documents))
print("Excluded before chunking:", len(excluded_for_chunking))


Chunkable pages: 319
Excluded before chunking: 13


Start with saving the final cleaned corpus state, so we can always reproduce it later.

This code is saving your cleaned PDF text into separate .txt files, one output file per original PDF.

### Code 026 — Start with saving the final cleaned corpus state, so we can always reproduce it later.


In [ ]:
import os
import pandas as pd
from collections import defaultdict

# Save final cleaned documents by source file
final_docs_by_file = defaultdict(list)

for doc in final_v2_documents:
    filename = doc.metadata.get("document_name")
    final_docs_by_file[filename].append(doc)

saved_final_cleaned_files = []

for filename, pages in final_docs_by_file.items():

    pages = sorted(
        pages,
        key=lambda x: x.metadata.get("pdf_page_number", 0)
    )

    output_name = (
        os.path.splitext(filename)[0]
        + "_FINAL_CLEANED_V2.txt"
    )

    output_path = os.path.join(
        CLEANED_FOLDER,
        output_name
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for page in pages:

            page_number = page.metadata.get(
                "pdf_page_number"
            )

            f.write(
                f"\n\n===== PDF PAGE {page_number} =====\n\n"
            )

            f.write(
                page.page_content
            )

    saved_final_cleaned_files.append(
        output_path
    )

print(
    "Saved final cleaned files:",
    len(saved_final_cleaned_files)
)

Saved final cleaned files: 7


Also save the excluded-page report:

### Code 027 — Also save the excluded-page report:


In [ ]:
excluded_for_chunking_df = pd.DataFrame(
    excluded_for_chunking
)

display(
    excluded_for_chunking_df
)

excluded_for_chunking_df.to_csv(
    os.path.join(
        EVALUATION_FOLDER,
        "pages_excluded_before_chunking.csv"
    ),
    index=False
)

,document,page,reason
0,Updated-PMOS-Guideline-12-June26.pdf,2,broken_spaced_text
1,Updated-PMOS-Guideline-12-June26.pdf,3,broken_spaced_text
2,Updated-PMOS-Guideline-12-June26.pdf,4,broken_spaced_text
3,Updated-PMOS-Guideline-12-June26.pdf,5,broken_spaced_text
4,Updated-PMOS-Guideline-12-June26.pdf,265,empty_page
5,Updated-PMOS-Guideline-12-June26.pdf,266,empty_page
6,Updated-PMOS-Guideline-12-June26.pdf,267,empty_page
7,Updated-PMOS-Guideline-12-June26.pdf,268,empty_page
8,Updated-PMOS-Guideline-12-June26.pdf,269,empty_page
9,Updated-PMOS-Guideline-12-June26.pdf,270,empty_page


Now quickly confirm the size:

### Code 028 — Now quickly confirm the size:


In [ ]:
print("All final cleaned pages:", len(final_v2_documents))
print("Chunkable pages:", len(chunkable_documents))
print("Excluded pages:", len(excluded_for_chunking))

All final cleaned pages: 332
Chunkable pages: 319
Excluded pages: 13


#Next: baseline recursive chunking

This will be our Method A.

Why first? Because we need a simple baseline to compare against semantic/SLM-style chunking later.

Install/import:

### Code 029 — This will be our Method A.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

Create the splitter:

### Code 030 — Create the splitter:


In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ],

    length_function=len
)

Now create chunks:

### Code 031 — Now create chunks:


In [ ]:
recursive_chunks = []

chunk_counter = 0

for doc in chunkable_documents:

    split_docs = recursive_splitter.split_documents(
        [doc]
    )

    for split_doc in split_docs:

        split_doc.metadata = doc.metadata.copy()

        split_doc.metadata[
            "chunking_method"
        ] = "recursive"

        split_doc.metadata[
            "chunk_id"
        ] = f"REC_CHUNK_{chunk_counter:05d}"

        split_doc.metadata[
            "character_count"
        ] = len(
            split_doc.page_content
        )

        split_doc.metadata[
            "word_count"
        ] = len(
            split_doc.page_content.split()
        )

        recursive_chunks.append(
            split_doc
        )

        chunk_counter += 1


print(
    "Recursive chunks created:",
    len(recursive_chunks)
)

Recursive chunks created: 1815


Now inspect a few:

### Code 032 — Now inspect a few:


In [ ]:
for chunk in recursive_chunks[:15]:

    print(
        "\nCHUNK ID:",
        chunk.metadata.get(
            "chunk_id"
        )
    )

    print(
        "Document:",
        chunk.metadata.get(
            "document_name"
        )
    )

    print(
        "Page:",
        chunk.metadata.get(
            "pdf_page_number"
        )
    )

    print(
        "Words:",
        chunk.metadata.get(
            "word_count"
        )
    )

    print()

    print(
        chunk.page_content
    )

    print(
        "=" * 100
    )


CHUNK ID: REC_CHUNK_00000
Document: NICHDWHPCOSweb.pdf
Page: 1
Words: 29

Eunice Kennedy Shriver National Institute of Child Health and Human Development (NICHD)

Polycystic Ovary Syndrome (PCOS) and Women's Health

Advancing Diagnosis and Treatments to Improve Outcomes

What is PCOS?

CHUNK ID: REC_CHUNK_00001
Document: NICHDWHPCOSweb.pdf
Page: 1
Words: 126

What is PCOS?

PCOS describes a group of symptoms related to a hormonal imbalance that can affect women and girls of reproductive age. A PCOS diagnosis means at least two of the following features are present: 1) sporadic ovulation, resulting in irregular or no menstrual periods; 2) high levels of androgen hormones, or signs of high androgen levels, such as acne that does not respond well to usual treatments; 3) growths in one or both ovaries, often clumps of ovarian follicles that have stopped developing. People with PCOS can also be at higher risk for infertility, diabetes and metabolic conditions, and cardiovascular disease. T

Then export all chunks to CSV,

### Code 033 — Then export all chunks to CSV,


In [ ]:
recursive_chunk_rows = []

for chunk in recursive_chunks:

    recursive_chunk_rows.append({

        "chunk_id":
            chunk.metadata.get(
                "chunk_id"
            ),

        "document_name":
            chunk.metadata.get(
                "document_name"
            ),

        "pdf_page_number":
            chunk.metadata.get(
                "pdf_page_number"
            ),

        "organization":
            chunk.metadata.get(
                "organization"
            ),

        "source_type":
            chunk.metadata.get(
                "source_type"
            ),

        "chunking_method":
            chunk.metadata.get(
                "chunking_method"
            ),

        "word_count":
            chunk.metadata.get(
                "word_count"
            ),

        "character_count":
            chunk.metadata.get(
                "character_count"
            ),

        "text":
            chunk.page_content
    })


recursive_chunks_df = pd.DataFrame(
    recursive_chunk_rows
)

display(
    recursive_chunks_df.head()
)

,chunk_id,document_name,pdf_page_number,organization,source_type,chunking_method,word_count,character_count,text
0,REC_CHUNK_00000,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,recursive,29,211,Eunice Kennedy Shriver National Institute of C...
1,REC_CHUNK_00001,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,recursive,126,783,What is PCOS?\n\nPCOS describes a group of sym...
2,REC_CHUNK_00002,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,recursive,89,623,How does NICHD support PCOS research?\n\nNICHD...
3,REC_CHUNK_00003,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,recursive,89,586,Success Snapshots\n\nClarifying the Role of Ge...
4,REC_CHUNK_00004,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,recursive,82,561,Defining Subtypes of PCOS\n\nBecause symptoms ...


### Code 034 — Prepare recursive_chunks_path


In [ ]:
recursive_chunks_path = os.path.join(
    CHUNKS_FOLDER,
    "recursive_chunks.csv"
)

recursive_chunks_df.to_csv(
    recursive_chunks_path,
    index=False
)

print(
    "Saved:",
    recursive_chunks_path
)

Saved: /content/drive/MyDrive/pcos_rag/chunks/recursive_chunks.csv


Before semantic chunking, let's also automatically identify suspicious chunks.

### Code 035 — Before semantic chunking, let's also automatically identify suspicious chunks.


In [ ]:
def check_chunk_quality(chunk):

    text = chunk.page_content.strip()

    problems = []

    words = text.split()

    if len(words) < 35:
        problems.append(
            "TOO_SHORT"
        )

    if "�" in text:
        problems.append(
            "ENCODING_ERROR"
        )

    if re.search(
        r"https?://|www\.",
        text,
        flags=re.I
    ):
        problems.append(
            "URL_REMAINS"
        )

    if len(words) >= 20:

        single_chars = sum(
            1
            for word in words
            if len(word) == 1
            and word.isalpha()
        )

        ratio = (
            single_chars /
            len(words)
        )

        if ratio >= 0.30:
            problems.append(
                "BROKEN_SPACED_TEXT"
            )

    return problems

### Code 036 — Define recursive_chunk_audit


In [ ]:
recursive_chunk_audit = []

for chunk in recursive_chunks:

    problems = check_chunk_quality(
        chunk
    )

    if problems:

        recursive_chunk_audit.append({

            "chunk_id":
                chunk.metadata.get(
                    "chunk_id"
                ),

            "document":
                chunk.metadata.get(
                    "document_name"
                ),

            "page":
                chunk.metadata.get(
                    "pdf_page_number"
                ),

            "problems":
                ", ".join(
                    problems
                ),

            "text":
                chunk.page_content
        })


recursive_chunk_audit_df = pd.DataFrame(
    recursive_chunk_audit
)

print(
    "Flagged recursive chunks:",
    len(recursive_chunk_audit_df)
)

display(
    recursive_chunk_audit_df.head(30)
)

Flagged recursive chunks: 420


,chunk_id,document,page,problems,text
0,REC_CHUNK_00000,NICHDWHPCOSweb.pdf,1,TOO_SHORT,Eunice Kennedy Shriver National Institute of C...
1,REC_CHUNK_00013,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,3,TOO_SHORT,Management of Polycystic Ovarian Syndrome\n\nM...
2,REC_CHUNK_00016,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,6,TOO_SHORT,National Guidance\n\nESRE 2018 NICE 2018 RCOG\...
3,REC_CHUNK_00019,PCOS-Guideline-Summary-2023-2.pdf,2,TOO_SHORT,"Evidence: Best practice, evidence-based guidel..."
4,REC_CHUNK_00024,PCOS-Guideline-Summary-2023-2.pdf,2,TOO_SHORT,. PCOS should be considered a high-risk condit...
5,REC_CHUNK_00036,PCOS-Guideline-Summary-2023-2.pdf,5,TOO_SHORT,Table 1: Categories of the PCOS guideline reco...
6,REC_CHUNK_00042,PCOS-Guideline-Summary-2023-2.pdf,6,TOO_SHORT,Table 2: Quality (certainty) of evidence categ...
7,REC_CHUNK_00053,PCOS-Guideline-Summary-2023-2.pdf,7,TOO_SHORT,| 1.1.5 | PP | Ovulatory dysfunction can still...
8,REC_CHUNK_00054,PCOS-Guideline-Summary-2023-2.pdf,8,TOO_SHORT,OOO⊕\n\n+\n\nOOO
9,REC_CHUNK_00055,PCOS-Guideline-Summary-2023-2.pdf,8,TOO_SHORT,|----------------|--------|-------------------...


Let's improve the audit first

Do not remove every <35 word chunk. Replace the current quality checker with this:

## Study-Specific Heuristic Parameters

The following are transparent **audit/filtering heuristics**, not validated universal constants:

- `< 35 words` → short-chunk review flag;
- `>= 0.30` single-character-token ratio → likely broken spaced text;
- `< 8 words` → extremely small retrieval fragment;
- `>= 2` reference-pattern hits → reference-heavy exclusion;
- SLM min/max character limits and prompt-context length → fixed implementation constraints.

They are used conservatively and logged; they are not described as literature-derived optimum values.


### Code 037 — Study-Specific Heuristic Parameters


In [ ]:
import re
import pandas as pd


def check_recursive_chunk(chunk):

    text = chunk.page_content.strip()
    words = text.split()

    problems = []

    # ------------------------------------------------
    # 1. Empty
    # ------------------------------------------------

    if not text:
        problems.append("EMPTY")
        return problems


    # ------------------------------------------------
    # 2. Very short
    # Only FLAG — do not automatically remove
    # ------------------------------------------------

    if len(words) < 35:
        problems.append("SHORT_REVIEW")


    # ------------------------------------------------
    # 3. Obvious title-only / structural chunks
    # ------------------------------------------------

    structural_patterns = [

        r"^Polycystic Ovary Syndrome \(PCOS\) \| ACOG$",

        r"^Polycystic ovary syndrome$",

        r"^International evidence-based\s+guideline.*?Summary$",

        r"^International Evidence-based\s+Guideline.*?2023$",

        r"^Chapter\s+(One|Two|Three|Four|Five|Six)\b",

    ]

    normalized = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    for pattern in structural_patterns:

        if re.search(
            pattern,
            normalized,
            flags=re.I
        ):
            problems.append(
                "STRUCTURAL_ONLY"
            )
            break


    # ------------------------------------------------
    # 4. Browser/navigation remnants
    # ------------------------------------------------

    navigation_terms = [
        "Related A-Z Topics",
        "About ACOG",
        "Referral Proforma",
        "Paragraph Styles",
        "Table Styles",
        "Copyright 2026",
    ]

    nav_hits = sum(
        term.lower() in text.lower()
        for term in navigation_terms
    )

    if nav_hits >= 1:
        problems.append(
            "NAVIGATION_OR_ADMIN"
        )


    # ------------------------------------------------
    # 5. Broken spaced text
    # ------------------------------------------------

    if len(words) >= 20:

        single_chars = sum(
            1
            for word in words
            if len(word) == 1
            and word.isalpha()
        )

        ratio = single_chars / len(words)

        if ratio >= 0.30:
            problems.append(
                "BROKEN_SPACED_TEXT"
            )


    # ------------------------------------------------
    # 6. Encoding issue
    # ------------------------------------------------

    if "�" in text:
        problems.append(
            "ENCODING_ERROR"
        )


    # ------------------------------------------------
    # 7. Remaining URL
    # ------------------------------------------------

    if re.search(
        r"https?://|www\.",
        text,
        flags=re.I
    ):
        problems.append(
            "URL_REMAINS"
        )


    # ------------------------------------------------
    # 8. Likely incomplete ending
    #
    # This is a REVIEW flag only.
    # ------------------------------------------------

    last_line = text.splitlines()[-1].strip()

    if (
        len(words) >= 35
        and last_line
        and not re.search(
            r'[.!?:;)"\]]$',
            last_line
        )
    ):
        problems.append(
            "POSSIBLE_INCOMPLETE_END"
        )


    # ------------------------------------------------
    # 9. Likely incomplete beginning
    # ------------------------------------------------

    first_line = text.splitlines()[0].strip()

    continuation_starters = (
        "and ",
        "or ",
        "but ",
        "which ",
        "while ",
        "whilst ",
        "making ",
        "including ",
        "of ",
        "to ",
        "with ",
    )

    if first_line.lower().startswith(
        continuation_starters
    ):
        problems.append(
            "POSSIBLE_INCOMPLETE_START"
        )


    return problems

Now create a new audit:

### Code 038 — Now create a new audit:


In [ ]:
recursive_review_rows = []

for chunk in recursive_chunks:

    problems = check_recursive_chunk(
        chunk
    )

    recursive_review_rows.append({

        "chunk_id":
            chunk.metadata.get(
                "chunk_id"
            ),

        "document":
            chunk.metadata.get(
                "document_name"
            ),

        "page":
            chunk.metadata.get(
                "pdf_page_number"
            ),

        "word_count":
            chunk.metadata.get(
                "word_count"
            ),

        "auto_flags":
            ", ".join(problems)
            if problems
            else "OK",

        "text":
            chunk.page_content,


    })


recursive_review_df = pd.DataFrame(
    recursive_review_rows
)

display(
    recursive_review_df.head(20)
)

,chunk_id,document,page,word_count,auto_flags,text
0,REC_CHUNK_00000,NICHDWHPCOSweb.pdf,1,29,SHORT_REVIEW,Eunice Kennedy Shriver National Institute of C...
1,REC_CHUNK_00001,NICHDWHPCOSweb.pdf,1,126,OK,What is PCOS?\n\nPCOS describes a group of sym...
2,REC_CHUNK_00002,NICHDWHPCOSweb.pdf,1,89,POSSIBLE_INCOMPLETE_END,How does NICHD support PCOS research?\n\nNICHD...
3,REC_CHUNK_00003,NICHDWHPCOSweb.pdf,1,89,POSSIBLE_INCOMPLETE_END,Success Snapshots\n\nClarifying the Role of Ge...
4,REC_CHUNK_00004,NICHDWHPCOSweb.pdf,1,82,OK,Defining Subtypes of PCOS\n\nBecause symptoms ...
5,REC_CHUNK_00005,NICHDWHPCOSweb.pdf,2,91,POSSIBLE_INCOMPLETE_END,Selected NICHD-Funded Polycystic Ovary Syndrom...
6,REC_CHUNK_00006,NICHDWHPCOSweb.pdf,2,72,POSSIBLE_INCOMPLETE_END,Defining PCOS Subtypes\n\nSeveral NICHD-funded...
7,REC_CHUNK_00007,NICHDWHPCOSweb.pdf,2,100,POSSIBLE_INCOMPLETE_END,Expanding Research Models\n\nHaving accurate r...
8,REC_CHUNK_00008,NICHDWHPCOSweb.pdf,2,101,POSSIBLE_INCOMPLETE_END,Treatment & Diagnostic Progress\n\nStudying th...
9,REC_CHUNK_00009,NICHDWHPCOSweb.pdf,2,95,OK,Identifying Early Diagnostic Measures\n\nDaugh...


### Code 039 — Prepare recursive_review_path


In [ ]:
recursive_review_path = os.path.join(
    EVALUATION_FOLDER,
    "recursive_chunks_review.csv"
)

recursive_review_df.to_csv(
    recursive_review_path,
    index=False
)

print(
    "Saved:",
    recursive_review_path
)

Saved: /content/drive/MyDrive/pcos_rag/evaluation/recursive_chunks_review.csv


Then see the flag distribution:

### Code 040 — Then see the flag distribution:


In [ ]:
from collections import Counter

flag_counter = Counter()

for flags in recursive_review_df["auto_flags"]:

    if flags == "OK":
        flag_counter["OK"] += 1

    else:

        for flag in flags.split(", "):
            flag_counter[flag] += 1


flag_summary_df = pd.DataFrame(
    flag_counter.items(),
    columns=[
        "flag",
        "count"
    ]
).sort_values(
    "count",
    ascending=False
)

display(flag_summary_df)

,flag,count
2,POSSIBLE_INCOMPLETE_END,847
1,OK,542
0,SHORT_REVIEW,420
4,POSSIBLE_INCOMPLETE_START,13
5,STRUCTURAL_ONLY,6
3,NAVIGATION_OR_ADMIN,4


That is more than half of the chunks. However, this flag is intentionally sensitive, so it does not mean 751 chunks are definitely bad. It means many chunks do not end with punctuation, which often happens when a character-based splitter cuts across a sentence or table line.

Recursive character-based chunking produced a large number of potentially incomplete boundaries, particularly at chunk endings, despite preserving many usable chunks. Therefore, a semantic boundary-aware method was evaluated as an alternative.

#Next step: build the actual SLM-assisted chunking method

Step A — Sentence units first

Before giving text to an SLM, we split each page into sentence/line units.

### Code 041 — Step A — Sentence units first


In [ ]:
import re


def create_atomic_units(text):

    if not text:
        return []

    # Split on:
    # sentence endings
    # or PDF line boundaries
    raw_units = re.split(
        r'(?<=[.!?])\s+|\n+',
        text.strip()
    )

    units = []

    for unit in raw_units:

        unit = unit.strip()

        if not unit:
            continue

        # Remove only excessive spaces
        unit = re.sub(
            r"\s+",
            " ",
            unit
        )

        units.append(unit)

    return units

### Code 042 — Prepare sample_doc


In [ ]:
sample_doc = chunkable_documents[20]

sample_units = create_atomic_units(
    sample_doc.page_content
)

print(
    "Atomic units:",
    len(sample_units)
)

for i, unit in enumerate(
    sample_units[:20]
):
    print(
        f"{i}: {unit}"
    )

Atomic units: 31
0: H
1: H
2: ○O⊕⊕
3: |----------------|--------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------|
4: | 1.8 | | Cardiovascular disease risk | |
5: | 1.8.1 | EBR | Women with PCOS should be considered at increased risk of cardiovascular disease and potentially of cardiovascular mortality, acknowledging that the overall risk of cardiovascular disease in premenopausal women is low.
6: | |
7: | 1.8.2 | EBR | All women with PCOS should be assessed for cardiovascular disease risk factors.
8: | |
9: | 1.8.3 | CR | All women with PCOS, regardless of age and BMI, should have a lipid profile (cholesterol, low density lipoprotein cholesterol, high density lipoprotei

Step B — Install the local SLM

### Code 043 — Step B — Install the local SLM


In [ ]:
!pip install -q transformers accelerate sentencepiece

### Code 044 — Imports and Setup


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

### Code 045 — Display Results


In [ ]:
print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

CUDA available: True
GPU: NVIDIA L4


Step C — Load the small language model

### Code 046 — Step C — Load the small language model


In [ ]:
SLM_MODEL_NAME = (
    "Qwen/Qwen2.5-1.5B-Instruct"
)

Load tokenizer:

### Code 047 — Load tokenizer:


In [ ]:
slm_tokenizer = AutoTokenizer.from_pretrained(
    SLM_MODEL_NAME
)

Load model:

### Code 048 — Load model:


In [ ]:
slm_model = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map=(
        "auto"
        if torch.cuda.is_available()
        else None
    )
)

if not torch.cuda.is_available():
    slm_model = slm_model.to("cpu")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Code 049 — Notebook Processing Step


In [ ]:
slm_model.eval()

print(
    "SLM loaded:",
    SLM_MODEL_NAME
)

SLM loaded: Qwen/Qwen2.5-1.5B-Instruct


Step D — What exactly will the SLM decide?

We should not ask:

“Create chunks from this document.”

That allows the model to rewrite text.

Instead ask it one specific question:

Should these two neighboring text units stay together or should there be a semantic boundary between them?

Step E — Boundary prompt

### Code 050 — Step E — Boundary prompt


In [ ]:
def build_boundary_prompt(
    previous_context,
    current_unit
):

    return f"""
You are identifying semantic chunk boundaries in an
evidence-based medical guideline.

Your task is ONLY to decide whether CURRENT TEXT
should remain in the same semantic chunk as
PREVIOUS CONTEXT.

Do not summarize.
Do not rewrite.
Do not correct medical wording.
Do not add any medical information.

Return exactly ONE label:

SAME

or

BREAK


Choose SAME when CURRENT TEXT:

- continues the same recommendation topic,
- provides another recommendation within the same subsection,
- provides details, cautions, explanations, or qualifications
  related to the previous idea,
- belongs to the same closely related clinical or lifestyle topic,
- is the first recommendation belonging to the heading immediately
  before it,
- continues a list or group of recommendations about the same subject.


Choose BREAK when CURRENT TEXT:

- begins a clearly different numbered subsection,
- introduces a new major medical, lifestyle, diagnostic,
  treatment, fertility, psychological, or other subject,
- is semantically independent from the previous context,
- clearly starts a new guideline topic.


IMPORTANT GUIDELINE RULES:

1. A new recommendation number does NOT automatically mean BREAK.

For example:

5.1.1 → 5.1.2

may be SAME because both belong to subsection 5.1.

2. A subsection heading and its first recommendation normally
belong together.

For example:

5.1 Preconception risk factors
→
5.1.1 Recommendation...

should normally be SAME.

3. Recommendations under the same subsection should normally
remain together unless they discuss clearly different concepts
and the accumulated chunk has become too broad.

4. A transition between different subsections should normally
be BREAK.

For example:

5.1 Preconception risk factors
→
5.2 Tubal patency testing

should be BREAK.


PREVIOUS CONTEXT:
{previous_context}


CURRENT TEXT:
{current_unit}


LABEL:
""".strip()

Step F — Ask the SLM for one boundary decision

### Code 051 — Step F — Ask the SLM for one boundary decision


In [ ]:
def slm_boundary_decision(
    previous_context,
    current_unit
):

    prompt = build_boundary_prompt(
        previous_context,
        current_unit
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = (
        slm_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )

    inputs = slm_tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(
            slm_model.device
        )
        for key, value
        in inputs.items()
    }

    with torch.no_grad():

        output = slm_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    generated = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    answer = slm_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    answer = answer.strip().upper()

    if "BREAK" in answer:
        return "BREAK"

    return "SAME"

Step G — Test the SLM before processing the whole corpus

Do not run it on 320 pages yet.

First test obvious examples.

### Code 052 — Step G — Test the SLM before processing the whole corpus


In [ ]:
test_1 = slm_boundary_decision(
    previous_context=(
        "Healthy lifestyle behaviours are important "
        "in women with PCOS."
    ),
    current_unit=(
        "Physical activity should be encouraged "
        "according to individual preferences."
    )
)

print(test_1)

SAME


### Code 053 — Prepare test_2


In [ ]:
test_2 = slm_boundary_decision(
    previous_context=(
        "Physical activity should be encouraged "
        "according to individual preferences."
    ),
    current_unit=(
        "Assessment of biochemical hyperandrogenism"
    )
)

print(test_2)

BREAK


Also test real corpus text:

### Code 054 — Also test real corpus text:


In [ ]:
sample_doc = chunkable_documents[100]

units = create_atomic_units(
    sample_doc.page_content
)

for i in range(
    1,
    min(len(units), 10)
):

    decision = slm_boundary_decision(
        previous_context=units[i - 1],
        current_unit=units[i]
    )

    print(
        "\nPREVIOUS:",
        units[i - 1]
    )

    print(
        "CURRENT:",
        units[i]
    )

    print(
        "DECISION:",
        decision
    )

    print("-" * 80)


PREVIOUS: 4++4
CURRENT: | No.
DECISION: BREAK
--------------------------------------------------------------------------------

PREVIOUS: | No.
CURRENT: Living # | Type | Recommendation | Grade/ Quality |
DECISION: BREAK
--------------------------------------------------------------------------------

PREVIOUS: Living # | Type | Recommendation | Grade/ Quality |
CURRENT: |----------------|--------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------|
DECISION: SAME
--------------------------------------------------------------------------------

PREVIOUS: |----------------|--------|---------------------

1. What is wrong with the current method?

Our old function did:
means:

every PDF line break may become a separate unit.

But PDF line breaks are often caused by page layout, not meaning.

So remove that approach.

2. Build proper semantic atomic units

We want this:

PDF extracted lines
        ↓
Merge wrapped lines
        ↓
Recognize:
- section headings
- recommendation numbers
- bullet points
- complete sentences
        ↓
SLM receives meaningful units

### Code 055 — 2. Build proper semantic atomic units


In [ ]:
import re


TABLE_HEADER_LINES = {
    "No.",
    "Living# Type Recommendation",
    "Grade/",
    "Quality",
    "Grade/ Quality"
}


def is_recommendation_start(line):
    """
    Detect recommendation IDs such as:
    5.1.1 EBR
    3.4.2 CR
    1.2.10 PP
    """

    return bool(
        re.match(
            r"^\d+(?:\.\d+){1,3}\s+"
            r"(EBR|CR|PP)\b",
            line,
            flags=re.I
        )
    )


def is_section_heading(line):
    """
    Detect headings such as:
    5.1# Preconception risk factors
    3 Lifestyle management
    4.2 Pharmacological treatment
    """

    return bool(
        re.match(
            r"^\d+(?:\.\d+)*#?\s+[A-Z]",
            line
        )
    )


def is_bullet_start(line):

    return bool(
        re.match(
            r"^[•●▪✓]\s*",
            line
        )
    )

Now the improved unit creator:

### Code 056 — Now the improved unit creator:


In [ ]:
def create_atomic_units_v2(text):

    if not text:
        return []

    # ---------------------------------
    # Get original PDF lines
    # ---------------------------------

    raw_lines = [
        re.sub(
            r"\s+",
            " ",
            line
        ).strip()

        for line in text.splitlines()

        if line.strip()
    ]


    # ---------------------------------
    # Remove only repeated table
    # scaffolding from CHUNK INPUT
    #
    # This does NOT modify our saved
    # cleaned source documents.
    # ---------------------------------

    lines = []

    for line in raw_lines:

        if line in TABLE_HEADER_LINES:
            continue

        lines.append(line)


    units = []

    current = ""


    for line in lines:

        # =================================
        # New recommendation
        # =================================

        if is_recommendation_start(line):

            if current:
                units.append(
                    current.strip()
                )

            current = line

            continue


        # =================================
        # New section heading
        # =================================

        if is_section_heading(line):

            if current:
                units.append(
                    current.strip()
                )

            current = line

            continue


        # =================================
        # New bullet
        # =================================

        if is_bullet_start(line):

            if current:
                units.append(
                    current.strip()
                )

            current = line

            continue


        # =================================
        # Normal PDF wrapped line
        # → join with existing text
        # =================================

        if current:

            current += " " + line

        else:

            current = line


        # =================================
        # If we now have a complete
        # sentence, close the unit.
        # =================================

        if re.search(
            r'[.!?]["\']?$',
            current
        ):

            units.append(
                current.strip()
            )

            current = ""


    if current:
        units.append(
            current.strip()
        )


    return units

3. Test it on the exact problematic page

Use the same document you just tested.

### Code 057 — 3. Test it on the exact problematic page


In [ ]:
sample_doc = chunkable_documents[100]

units = create_atomic_units_v2(
    sample_doc.page_content
)

print(
    "Atomic units:",
    len(units)
)

for i, unit in enumerate(
    units
):
    print(
        f"\nUNIT {i}"
    )

    print(unit)

    print("-" * 80)

Atomic units: 1

UNIT 0
4++4 | No. Living # | Type | Recommendation | Grade/ Quality | |----------------|--------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------| | 5.1# | | Preconception risk factors | | | 5.1.1 | EBR | Women with should be counseled on the adverse impact of excess weight on clinical pregnancy, miscarriage and live birth rates, following infertility treatment. | | | 5.1.2 | CR | Consistent with routine preconception care, in women with PMOS/PCOS planning pregnancy, weight, blood pressure, smoking, alcohol, diet and nutritional status, folate supplementation (higher dose in those wi

### Code 058 — Run Iterative Processing


In [ ]:
for i in range(1, len(units)):

    previous_context = units[i - 1]
    current_unit = units[i]

    decision = slm_boundary_decision(
        previous_context=previous_context,
        current_unit=current_unit
    )

    print("\nPREVIOUS:")
    print(previous_context)

    print("\nCURRENT:")
    print(current_unit)

    print("\nDECISION:", decision)

    print("=" * 100)

So we'll make this a hybrid structural + SLM chunker.

### Code 059 — So we'll make this a hybrid structural + SLM chunker.


In [ ]:
def extract_section_number(text):
    """
    Examples:

    5.1# Preconception risk factors -> 5.1
    5.1.1 EBR ...                -> 5.1.1
    3.4.2 CR ...                 -> 3.4.2
    """

    match = re.match(
        r"^(\d+(?:\.\d+)+)",
        text.strip()
    )

    if match:
        return match.group(1)

    return None

### Code 060 — Recommendation:


In [ ]:
def get_parent_section(section_number):

    if not section_number:
        return None

    parts = section_number.split(".")

    # Recommendation:
    # 5.1.1 -> 5.1
    if len(parts) >= 3:
        return ".".join(parts[:-1])

    # Heading:
    # 5.1 -> 5.1
    return section_number

### Code 061 — Define examples


In [ ]:
examples = [
    "5.1# Preconception risk factors",
    "5.1.1 EBR Women with...",
    "5.1.2 CR Consistent with...",
    "5.2 Tubal patency testing",
    "5.3.1 EBR Letrozole..."
]

for text in examples:

    section = extract_section_number(text)

    print(
        text,
        "=>",
        section,
        "=> parent:",
        get_parent_section(section)
    )

5.1# Preconception risk factors => 5.1 => parent: 5.1
5.1.1 EBR Women with... => 5.1.1 => parent: 5.1
5.1.2 CR Consistent with... => 5.1.2 => parent: 5.1
5.2 Tubal patency testing => 5.2 => parent: 5.2
5.3.1 EBR Letrozole... => 5.3.1 => parent: 5.3


3. Create the improved boundary controller

Now instead of calling the SLM blindly for every pair, use

### Code 062 — 3. Create the improved boundary controller


In [ ]:
def hybrid_boundary_decision(
    previous_context,
    current_unit
):

    prev_section = extract_section_number(
        previous_context
    )

    curr_section = extract_section_number(
        current_unit
    )

    prev_parent = get_parent_section(
        prev_section
    )

    curr_parent = get_parent_section(
        curr_section
    )

    # -------------------------------------
    # RULE 1
    # Same numbered subsection
    #
    # 5.1 -> 5.1.1
    # 5.1.1 -> 5.1.2
    # -------------------------------------

    if (
        prev_parent is not None
        and curr_parent is not None
        and prev_parent == curr_parent
    ):
        return "SAME"


    # -------------------------------------
    # RULE 2
    # Clearly different numbered subsection
    #
    # 5.1 -> 5.2
    # 5.2.1 -> 5.3
    # -------------------------------------

    if (
        prev_parent is not None
        and curr_parent is not None
        and prev_parent != curr_parent
    ):
        return "BREAK"


    # -------------------------------------
    # RULE 3
    # Ambiguous normal prose / bullets /
    # unnumbered content
    # → ask SLM
    # -------------------------------------

    return slm_boundary_decision(
        previous_context=
            previous_context,

        current_unit=
            current_unit
    )

4. Retest your exact 12 units

### Code 063 — 4. Retest your exact 12 units


In [ ]:
for i in range(
    1,
    len(units)
):

    previous_context = (
        units[i - 1]
    )

    current_unit = (
        units[i]
    )

    decision = hybrid_boundary_decision(
        previous_context=
            previous_context,

        current_unit=
            current_unit
    )

    print(
        "\nPREVIOUS:"
    )

    print(
        previous_context
    )

    print(
        "\nCURRENT:"
    )

    print(
        current_unit
    )

    print(
        "\nDECISION:",
        decision
    )

    print(
        "=" * 100
    )

Why this is better than forcing the SLM to decide everything

Your documents are structured clinical guidelines. The numbering itself contains reliable semantic information.

So the final method should not be:

“SLM looks at every pair and guesses where to split.”

Instead:

“Guideline structural information is first used to preserve recommendations belonging to the same subsection, while the SLM resolves semantic boundaries where explicit structural cues are unavailable.”

That is a much stronger methodology.

##Use this chunk builder:

### Code 064 — Use this chunk builder:


In [ ]:
# Study-specific implementation constraints.
# Fixed for reproducibility; not claimed as universal optima.
SLM_MIN_CHARS = 250
SLM_MAX_CHARS = 1200
SLM_PROMPT_CONTEXT_CHARS = 1800


def build_slm_chunks_for_page(
    text,
    min_chars=SLM_MIN_CHARS,
    max_chars=SLM_MAX_CHARS,
    prompt_context_chars=SLM_PROMPT_CONTEXT_CHARS
):
    units = create_atomic_units_v2(text)

    if not units:
        return []

    chunks = []
    current_units = [units[0]]
    current_length = len(units[0])

    for unit in units[1:]:
        proposed_length = current_length + 1 + len(unit)

        if proposed_length > max_chars and current_length >= min_chars:
            chunks.append(" ".join(current_units))
            current_units = [unit]
            current_length = len(unit)
            continue

        if current_length < min_chars:
            current_units.append(unit)
            current_length = current_length + 1 + len(unit)
            continue

        previous_context = " ".join(current_units)
        previous_context = previous_context[-prompt_context_chars:]

        decision = hybrid_boundary_decision(
            previous_context=previous_context,
            current_unit=unit
        )

        if decision == "BREAK":
            chunks.append(" ".join(current_units))
            current_units = [unit]
            current_length = len(unit)
        else:
            current_units.append(unit)
            current_length = current_length + 1 + len(unit)

    if current_units:
        chunks.append(" ".join(current_units))

    return chunks


Now generate the full SLM-assisted corpus:

### Code 065 — Now generate the full SLM-assisted corpus:


In [ ]:
from langchain_core.documents import Document

slm_chunks = []

slm_counter = 0

for page_index, doc in enumerate(
    chunkable_documents,
    start=1
):

    page_chunks = build_slm_chunks_for_page(
        doc.page_content
    )

    for local_index, chunk_text in enumerate(
        page_chunks
    ):

        metadata = doc.metadata.copy()

        metadata.update({
            "chunking_method":
                "slm_assisted",

            "chunk_id":
                f"SLM_CHUNK_{slm_counter:05d}",

            "page_chunk_index":
                local_index,

            "character_count":
                len(chunk_text),

            "word_count":
                len(
                    chunk_text.split()
                )
        })

        slm_chunks.append(
            Document(
                page_content=chunk_text,
                metadata=metadata
            )
        )

        slm_counter += 1

    if page_index % 25 == 0:
        print(
            f"Processed {page_index}/"
            f"{len(chunkable_documents)} pages"
        )


print(
    "\nSLM-assisted chunks created:",
    len(slm_chunks)
)

Processed 25/319 pages
Processed 50/319 pages
Processed 75/319 pages
Processed 100/319 pages
Processed 125/319 pages
Processed 150/319 pages
Processed 175/319 pages
Processed 200/319 pages
Processed 225/319 pages
Processed 250/319 pages
Processed 275/319 pages
Processed 300/319 pages

SLM-assisted chunks created: 959


Then create the export table:

### Code 066 — Then create the export table:


In [ ]:
slm_chunk_rows = []

for chunk in slm_chunks:

    slm_chunk_rows.append({

        "chunk_id":
            chunk.metadata.get(
                "chunk_id"
            ),

        "document_name":
            chunk.metadata.get(
                "document_name"
            ),

        "pdf_page_number":
            chunk.metadata.get(
                "pdf_page_number"
            ),

        "page_chunk_index":
            chunk.metadata.get(
                "page_chunk_index"
            ),

        "organization":
            chunk.metadata.get(
                "organization"
            ),

        "source_type":
            chunk.metadata.get(
                "source_type"
            ),

        "chunking_method":
            chunk.metadata.get(
                "chunking_method"
            ),

        "word_count":
            chunk.metadata.get(
                "word_count"
            ),

        "character_count":
            chunk.metadata.get(
                "character_count"
            ),

        "text":
            chunk.page_content
    })


slm_chunks_df = pd.DataFrame(
    slm_chunk_rows
)

display(
    slm_chunks_df.head(10)
)

,chunk_id,document_name,pdf_page_number,page_chunk_index,organization,source_type,chunking_method,word_count,character_count,text
0,SLM_CHUNK_00000,NICHDWHPCOSweb.pdf,1,0,NICHD,patient_health_information,slm_assisted,146,938,Eunice Kennedy Shriver National Institute of C...
1,SLM_CHUNK_00001,NICHDWHPCOSweb.pdf,1,1,NICHD,patient_health_information,slm_assisted,80,562,How does NICHD support PCOS research? NICHD ha...
2,SLM_CHUNK_00002,NICHDWHPCOSweb.pdf,1,2,NICHD,patient_health_information,slm_assisted,85,557,Success Snapshots Clarifying the Role of Genet...
3,SLM_CHUNK_00003,NICHDWHPCOSweb.pdf,1,3,NICHD,patient_health_information,slm_assisted,82,560,Defining Subtypes of PCOS Because symptoms and...
4,SLM_CHUNK_00004,NICHDWHPCOSweb.pdf,2,0,NICHD,patient_health_information,slm_assisted,88,647,Selected NICHD-Funded Polycystic Ovary Syndrom...
5,SLM_CHUNK_00005,NICHDWHPCOSweb.pdf,2,1,NICHD,patient_health_information,slm_assisted,69,564,Defining PCOS Subtypes Several NICHD-funded pr...
6,SLM_CHUNK_00006,NICHDWHPCOSweb.pdf,2,2,NICHD,patient_health_information,slm_assisted,66,450,Expanding Research Models Having accurate rese...
7,SLM_CHUNK_00007,NICHDWHPCOSweb.pdf,2,3,NICHD,patient_health_information,slm_assisted,118,784,Learn More About NICHD PCOS Projects NICHD's P...
8,SLM_CHUNK_00008,NICHDWHPCOSweb.pdf,2,4,NICHD,patient_health_information,slm_assisted,95,647,Identifying Early Diagnostic Measures Daughter...
9,SLM_CHUNK_00009,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,1,0,"Saint Mary's Managed Clinical Service, Manches...",clinical_patient_guidance,slm_assisted,54,379,- Pregnancy - Hypothyroidism - Hyperprolactino...


### Code 067 — Prepare slm_chunks_path


In [ ]:
slm_chunks_path = os.path.join(
    CHUNKS_FOLDER,
    "slm_assisted_chunks.csv"
)

slm_chunks_df.to_csv(
    slm_chunks_path,
    index=False
)

print(
    "Saved:",
    slm_chunks_path
)

Saved: /content/drive/MyDrive/pcos_rag/chunks/slm_assisted_chunks.csv


Now run the same audit we used for recursive chunks:

### Code 068 — Now run the same audit we used for recursive chunks:


In [ ]:
slm_review_rows = []

for chunk in slm_chunks:

    problems = check_recursive_chunk(
        chunk
    )

    slm_review_rows.append({

        "chunk_id":
            chunk.metadata.get(
                "chunk_id"
            ),

        "document":
            chunk.metadata.get(
                "document_name"
            ),

        "page":
            chunk.metadata.get(
                "pdf_page_number"
            ),

        "word_count":
            chunk.metadata.get(
                "word_count"
            ),

        "auto_flags":
            ", ".join(problems)
            if problems
            else "OK",

        "text":
            chunk.page_content,

        "meaning_complete":
            "",

        "single_clear_topic":
            "",

        "context_sufficient":
            "",

        "keep_for_rag":
            "",

        "review_notes":
            ""
    })


slm_review_df = pd.DataFrame(
    slm_review_rows
)

Then get the flag summary:

### Code 069 — Then get the flag summary:


In [ ]:
from collections import Counter

slm_flag_counter = Counter()

for flags in slm_review_df[
    "auto_flags"
]:

    if flags == "OK":
        slm_flag_counter["OK"] += 1

    else:

        for flag in flags.split(", "):
            slm_flag_counter[
                flag
            ] += 1


slm_flag_summary_df = pd.DataFrame(
    slm_flag_counter.items(),
    columns=[
        "flag",
        "count"
    ]
).sort_values(
    "count",
    ascending=False
)

display(
    slm_flag_summary_df
)

,flag,count
0,OK,751
1,POSSIBLE_INCOMPLETE_END,145
2,SHORT_REVIEW,54
4,POSSIBLE_INCOMPLETE_START,8
5,STRUCTURAL_ONLY,6
3,NAVIGATION_OR_ADMIN,5


### Code 070 — Prepare slm_review_path


In [ ]:
slm_review_path = os.path.join(
    EVALUATION_FOLDER,
    "slm_chunks_review.csv"
)

slm_review_df.to_csv(
    slm_review_path,
    index=False
)

print(
    "Saved:",
    slm_review_path
)

Saved: /content/drive/MyDrive/pcos_rag/evaluation/slm_chunks_review.csv


# AI-Assisted Chunk Quality Evaluation

A reproducible **fixed-seed random sample** is taken separately from the Recursive and SLM-assisted corpora:

- 100 Recursive chunks
- 100 SLM-assisted chunks
- 200 sampled chunks in total

The scoring is **AI-assisted rubric evaluation** (e.g., ChatGPT/LLM applying a predefined rubric), not independent human gold-standard annotation. It is supplementary qualitative evidence and should be researcher spot-checked.

The final chunking decision is also supported by downstream retrieval metrics.

`random_state = 42` is used only for reproducibility.


### Code 071 — AI-Assisted Chunk Quality Evaluation


In [ ]:
RANDOM_SEED = 42
SAMPLE_SIZE = 100

Now sample recursive chunks:

### Code 072 — Now sample recursive chunks:


In [ ]:
recursive_rubric_sample = (
    recursive_review_df
    .sample(
        n=min(SAMPLE_SIZE, len(recursive_review_df)),
        random_state=RANDOM_SEED
    )
    .copy()
)


Sample SLM chunks:

### Code 073 — Sample SLM chunks:


In [ ]:
slm_rubric_sample = (
    slm_review_df
    .sample(
        n=min(SAMPLE_SIZE, len(slm_review_df)),
        random_state=RANDOM_SEED
    )
    .copy()
)


Add the method labels:

### Code 074 — Add the method labels:


In [ ]:
recursive_rubric_sample["evaluation_method"] = "recursive"
slm_rubric_sample["evaluation_method"] = "slm_assisted"


### Code 075 — Prepare ai_assisted_chunk_evaluation_df


In [ ]:
ai_assisted_chunk_evaluation_df = pd.concat(
    [recursive_rubric_sample, slm_rubric_sample],
    ignore_index=True
)


Add the predefined rubric columns. These are **AI-assisted reference scores**, not independently validated human annotations.

### Code 076 — Run Iterative Processing


In [ ]:
for col in [
    "semantic_completeness",
    "topic_coherence",
    "context_sufficiency",
    "boundary_quality",
    "noise_free",
    "overall_keep",
    "review_notes"
]:
    ai_assisted_chunk_evaluation_df[col] = ""

ai_assisted_chunk_evaluation_df["annotation_method"] = (
    "AI-assisted predefined rubric; researcher spot-check recommended"
)


### Code 077 — Prepare ai_rubric_path


In [ ]:
ai_rubric_path = os.path.join(
    EVALUATION_FOLDER,
    "ai_assisted_chunk_evaluation_200.csv"
)

ai_assisted_chunk_evaluation_df.to_csv(
    ai_rubric_path,
    index=False
)

print("Saved AI-assisted rubric template:", ai_rubric_path)


Saved AI-assisted rubric template: /content/drive/MyDrive/pcos_rag/evaluation/ai_assisted_chunk_evaluation_200.csv


## Rubric Used for AI-Assisted Chunk Evaluation

Each criterion uses a 0/1/2 ordinal scale. The rubric is study-specific and is not presented as a previously standardized chunking instrument.

Semantic completeness

**Semantic completeness**
- 0 = clearly incomplete meaning
- 1 = mostly understandable but context is missing
- 2 = complete idea

**Topic coherence**
- 0 = unrelated topics mixed
- 1 = mostly one topic
- 2 = single clear topic

**Context sufficiency**
- 0 = cannot be understood without surrounding text
- 1 = understandable but more context would help
- 2 = sufficient context to stand alone

**Boundary quality**
- 0 = starts/ends badly
- 1 = minor boundary problem
- 2 = clean beginning and ending

**Noise-free**
- 0 = strong navigation/header/table noise
- 1 = minor noise
- 2 = clean

The five dimensions have a maximum combined raw score of 10. The **Boundary Quality Score (BQS)** intentionally uses only the first four semantic/boundary dimensions. `noise_free` is reported separately and is **not** included in BQS.


### Code 078 — **Semantic completeness**


In [ ]:
import os

ai_scored_path = os.path.join(
    EVALUATION_FOLDER,
    "ai_assisted_chunk_evaluation_200_scored.csv"
)

scored_rubric_path = None

if os.path.exists(ai_scored_path):
    scored_rubric_path = ai_scored_path
else:
    # Neither scored file exists, prompt for template download and then upload
    template_path = os.path.join(EVALUATION_FOLDER, "ai_assisted_chunk_evaluation_200.csv")
    if not os.path.exists(template_path):
        raise FileNotFoundError(
            f"The template '{template_path}' does not exist. "
            "Please ensure the cell that creates this template (Code 077) has been run."
        )

    print(
        "No scored rubric file found. Please perform the AI-assisted scoring:\n"
        f"1. Download the template: {template_path} (from {EVALUATION_FOLDER})\n"
        "2. Score the chunks using your preferred AI model or manually.\n"
        "3. Save the scored file as 'ai_assisted_chunk_evaluation_200_scored.csv'.\n"
        "4. Upload the scored CSV file below to proceed."
    )

    # Use files.upload() to wait for user input
    from google.colab import files
    uploaded_files = files.upload()

    if not uploaded_files:
        raise RuntimeError("No file was uploaded. Please upload the scored CSV to continue.")

    uploaded_filename = list(uploaded_files.keys())[0]
    expected_filename = "ai_assisted_chunk_evaluation_200_scored.csv"

    if uploaded_filename != expected_filename:
        print(
            f"Warning: Uploaded file name '{uploaded_filename}' does not match "
            f"expected file name '{expected_filename}'. "
            "Please rename your file and re-upload, or place it directly in "
            f"'{EVALUATION_FOLDER}' with the correct name."
        )
        # Attempt to proceed with the uploaded file if it's a CSV, otherwise raise error.
        if uploaded_filename.lower().endswith(".csv"):
            # Save the uploaded file to the expected path in EVALUATION_FOLDER
            with open(ai_scored_path, "wb") as f:
                f.write(uploaded_files[uploaded_filename])
            scored_rubric_path = ai_scored_path
            print(f"Proceeding by saving uploaded file to: {ai_scored_path}. Please ensure it is correctly scored.")
        else:
            raise ValueError("Uploaded file is not a CSV. Please upload a CSV file.")
    else:
        # Save the uploaded file to the expected path in EVALUATION_FOLDER
        with open(ai_scored_path, "wb") as f:
            f.write(uploaded_files[uploaded_filename])
        scored_rubric_path = ai_scored_path
        print(f"Successfully uploaded and saved '{uploaded_filename}' to '{ai_scored_path}'.")


if scored_rubric_path:
    ai_assisted_chunk_evaluation_df = pd.read_csv(scored_rubric_path)
    display(ai_assisted_chunk_evaluation_df.head())
else:
    raise RuntimeError("Could not determine a path for the scored rubric file.")

No scored rubric file found. Please perform the AI-assisted scoring:
1. Download the template: /content/drive/MyDrive/pcos_rag/evaluation/ai_assisted_chunk_evaluation_200.csv (from /content/drive/MyDrive/pcos_rag/evaluation)
2. Score the chunks using your preferred AI model or manually.
3. Save the scored file as 'ai_assisted_chunk_evaluation_200_scored.csv'.
4. Upload the scored CSV file below to proceed.


Saving ai_assisted_chunk_evaluation_200_scored.csv to ai_assisted_chunk_evaluation_200_scored.csv
Successfully uploaded and saved 'ai_assisted_chunk_evaluation_200_scored.csv' to '/content/drive/MyDrive/pcos_rag/evaluation/ai_assisted_chunk_evaluation_200_scored.csv'.


,chunk_id,document,page,word_count,auto_flags,text,evaluation_method,meaning_complete,single_clear_topic,context_sufficient,...,review_notes,semantic_completeness,topic_coherence,context_sufficiency,boundary_quality,noise_free,raw_score_10,BQS,overall_keep,annotation_method
0,REC_CHUNK_00567,Updated-PMOS-Guideline-12-June26.pdf,60,80,POSSIBLE_INCOMPLETE_END,Hyperandrogenism is a key diagnostic feature o...,recursive,NaN,NaN,NaN,...,Ending lacks a clean terminal boundary.,1,2,2,1,2,8,6,NaN,AI-assisted predefined rubric; researcher spot...
1,REC_CHUNK_00432,Updated-PMOS-Guideline-12-June26.pdf,38,6,SHORT_REVIEW,H\n\nH\n\n出\n\n出\n\n出\n\n出,recursive,NaN,NaN,NaN,...,OCR/symbol noise; no usable semantic content.,0,0,0,0,0,0,0,NaN,AI-assisted predefined rubric; researcher spot...
2,REC_CHUNK_01750,Updated-PMOS-Guideline-12-June26.pdf,255,58,OK,"144. Pedersen O, Gaede P. Intensified multifac...",recursive,NaN,NaN,NaN,...,Reference-only chunk; citation boundaries are ...,1,1,1,2,2,7,5,NaN,AI-assisted predefined rubric; researcher spot...
3,REC_CHUNK_00674,Updated-PMOS-Guideline-12-June26.pdf,74,63,POSSIBLE_INCOMPLETE_END,| 1.8# | | Cardiovascular disease risk |\n|---...,recursive,NaN,NaN,NaN,...,Structured table/list; table markup is counted...,2,2,2,2,1,9,8,NaN,AI-assisted predefined rubric; researcher spot...
4,REC_CHUNK_00111,PCOS-Guideline-Summary-2023-2.pdf,17,52,POSSIBLE_INCOMPLETE_END,| 2.6.3.1 | CR | Public health actors should c...,recursive,NaN,NaN,NaN,...,Structured table/list; table markup is counted...,2,2,2,2,1,9,8,NaN,AI-assisted predefined rubric; researcher spot...


Convert rubric columns to numeric before calculating the study-specific Boundary Quality Score.

### Code 079 — Define boundary_cols


In [ ]:
boundary_cols = [
    "semantic_completeness",
    "topic_coherence",
    "context_sufficiency",
    "boundary_quality"
]

for col in boundary_cols + ["noise_free"]:
    ai_assisted_chunk_evaluation_df[col] = pd.to_numeric(
        ai_assisted_chunk_evaluation_df[col],
        errors="coerce"
    )

missing_boundary_scores = (
    ai_assisted_chunk_evaluation_df[boundary_cols]
    .isna()
    .any(axis=1)
    .sum()
)

print("Rows with missing BQS inputs:", missing_boundary_scores)


Rows with missing BQS inputs: 0


## Boundary Quality Score (BQS)

BQS is retained as a **study-specific normalized rubric score**:

\[
\mathrm{BQS}_i =
\frac{s_{SC,i}+s_{TC,i}+s_{CS,i}+s_{BQ,i}}{8}
\]

where \(s_{SC}\), \(s_{TC}\), \(s_{CS}\), and \(s_{BQ}\) are the semantic-completeness, topic-coherence, context-sufficiency, and boundary-quality scores. Each component ranges from 0 to 2, so the maximum numerator is \(4\times2=8\), normalizing BQS to \([0,1]\).

**BQS is not claimed as a published standard metric.** It is supplementary to established downstream retrieval evaluation.


### Code 080 — Boundary Quality Score (BQS)


In [ ]:
ai_assisted_chunk_evaluation_df["boundary_quality_score"] = (
    ai_assisted_chunk_evaluation_df[boundary_cols].sum(axis=1) / 8.0
)

bqs_summary = (
    ai_assisted_chunk_evaluation_df
    .groupby("evaluation_method")["boundary_quality_score"]
    .mean()
)

display(bqs_summary.round(3))


,boundary_quality_score
evaluation_method,
recursive,0.708
slm_assisted,0.900


Create the AI-assisted rubric comparison summary.

### Code 081 — Create the AI-assisted rubric comparison summary.


In [ ]:
rubric_comparison_summary = (
    ai_assisted_chunk_evaluation_df
    .groupby("evaluation_method")[
        [
            "semantic_completeness",
            "topic_coherence",
            "context_sufficiency",
            "boundary_quality",
            "noise_free",
            "boundary_quality_score"
        ]
    ]
    .mean()
    .round(3)
)

display(rubric_comparison_summary)


,semantic_completeness,topic_coherence,context_sufficiency,boundary_quality,noise_free,boundary_quality_score
evaluation_method,,,,,,
recursive,1.27,1.73,1.32,1.34,1.45,0.708
slm_assisted,1.77,1.84,1.71,1.88,1.67,0.900


### Code 082 — Prepare rubric_summary_path


In [ ]:
rubric_summary_path = os.path.join(
    EVALUATION_FOLDER,
    "ai_assisted_chunking_rubric_summary.csv"
)

rubric_comparison_summary.to_csv(rubric_summary_path)
print("AI-assisted chunking rubric summary saved:", rubric_summary_path)


AI-assisted chunking rubric summary saved: /content/drive/MyDrive/pcos_rag/evaluation/ai_assisted_chunking_rubric_summary.csv


Calculate BQS improvement:

\[
\mathrm{Relative\ Improvement} =
\frac{\mathrm{BQS}_{SLM}-\mathrm{BQS}_{Recursive}}
{\mathrm{BQS}_{Recursive}}\times100
\]


### Code 083 — Calculate BQS improvement:


In [ ]:
recursive_score = rubric_comparison_summary.loc[
    "recursive",
    "boundary_quality_score"
]

slm_score = rubric_comparison_summary.loc[
    "slm_assisted",
    "boundary_quality_score"
]

absolute_improvement = slm_score - recursive_score
relative_improvement = (absolute_improvement / recursive_score) * 100

print("Recursive BQS:", round(recursive_score, 3))
print("SLM-assisted BQS:", round(slm_score, 3))
print("Absolute BQS improvement:", round(absolute_improvement, 3))
print("Relative BQS improvement (%):", round(relative_improvement, 2))


Recursive BQS: 0.708
SLM-assisted BQS: 0.9
Absolute BQS improvement: 0.192
Relative BQS improvement (%): 27.12


## Retrieval Eligibility Filtering

Clearly unsuitable chunks are filtered before retrieval (structural-only titles, navigation/admin content, reference-heavy chunks, and extremely tiny fragments).

Thresholds such as `< 8 words` or `>= 2 reference-pattern hits` are **study-specific conservative filtering rules**, not universal/literature-derived optima. Exclusions are logged and inspected.


### Code 084 — Retrieval Eligibility Filtering


In [ ]:
import re

REFERENCE_PATTERNS = [
    r"\bReferences\b",
    r"\bCitations\b",
    r"\bPMID\b",
    r"\bRetrieved\s+\w+\s+\d{1,2},?\s+\d{4}\b",
    r"\bdoi\b",
]

ADMIN_PATTERNS = [
    r"Paragraph Styles",
    r"Table Styles",
    r"Copyright",
    r"Privacy Statement",
    r"Terms and Conditions",
    r"Related A-Z Topics",
    r"About ACOG",
    r"Referral Proforma",
    r"NICHD News and Features",
]

METADATA_NOISE_PATTERNS = [
    r"\bcontent last updated\b",
    r"\blast updated\b",
    r"\blast reviewed\b",
    r"\bpage last reviewed\b",
    r"\bupdated on\b",
    r"\bdate last updated\b",
]

STRUCTURAL_ONLY_PATTERNS = [
    r"^Polycystic Ovary Syndrome \(PCOS\) \| ACOG$",
    r"^Polycystic ovary syndrome$",
    r"^International evidence-based guideline.*Summary$",
    r"^International Evidence-based Guideline.*2023$",
]

### Code 085 — structural only


In [ ]:
def assess_retrieval_eligibility(text):

    if not text or not text.strip():
        return False, "empty"

    clean_text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    words = clean_text.split()

    # structural only
    for pattern in STRUCTURAL_ONLY_PATTERNS:
        if re.search(
            pattern,
            clean_text,
            flags=re.I
        ):
            return False, "structural_only"

    # admin / navigation
    admin_hits = sum(
        1
        for pattern in ADMIN_PATTERNS
        if re.search(
            pattern,
            clean_text,
            flags=re.I
        )
    )

    if admin_hits >= 1:
        return False, "navigation_or_admin"

    # metadata / footer noise
    for pattern in METADATA_NOISE_PATTERNS:
        if re.search(
            pattern,
            clean_text,
            flags=re.I
        ):
            return False, "metadata_or_footer_noise"

    # reference-heavy
    reference_hits = sum(
        1
        for pattern in REFERENCE_PATTERNS
        if re.search(
            pattern,
            clean_text,
            flags=re.I
        )
    )

    if reference_hits >= 2:
        return False, "reference_heavy"

    # only remove extremely tiny fragments
    if len(words) < 8:
        return False, "too_short_to_retrieve"

    return True, "eligible"

Apply to recursive chunks:

### Code 086 — Apply to recursive chunks:


In [ ]:
recursive_retrieval_chunks = []
recursive_excluded_retrieval = []

for chunk in recursive_chunks:

    eligible, reason = assess_retrieval_eligibility(
        chunk.page_content
    )

    if eligible:
        recursive_retrieval_chunks.append(chunk)

    else:
        recursive_excluded_retrieval.append({
            "chunk_id": chunk.metadata.get("chunk_id"),
            "document": chunk.metadata.get("document_name"),
            "page": chunk.metadata.get("pdf_page_number"),
            "reason": reason,
            "text": chunk.page_content
        })

print(
    "Recursive retrieval chunks:",
    len(recursive_retrieval_chunks)
)

print(
    "Recursive excluded:",
    len(recursive_excluded_retrieval)
)

Recursive retrieval chunks: 1584
Recursive excluded: 231


Then apply to SLM chunks:

### Code 087 — Then apply to SLM chunks:


In [ ]:
slm_retrieval_chunks = []
slm_excluded_retrieval = []

for chunk in slm_chunks:

    eligible, reason = assess_retrieval_eligibility(
        chunk.page_content
    )

    if eligible:
        slm_retrieval_chunks.append(chunk)

    else:
        slm_excluded_retrieval.append({
            "chunk_id": chunk.metadata.get("chunk_id"),
            "document": chunk.metadata.get("document_name"),
            "page": chunk.metadata.get("pdf_page_number"),
            "reason": reason,
            "text": chunk.page_content
        })

print(
    "SLM retrieval chunks:",
    len(slm_retrieval_chunks)
)

print(
    "SLM excluded:",
    len(slm_excluded_retrieval)
)

SLM retrieval chunks: 932
SLM excluded: 27


The next step should not be FAISS immediately. First, inspect the exclusion reasons so we make sure the filter did not remove useful medical recommendations.

### Code 088 — Imports and Setup


In [ ]:
from collections import Counter
import pandas as pd

recursive_exclusion_summary = Counter(
    item["reason"]
    for item in recursive_excluded_retrieval
)

slm_exclusion_summary = Counter(
    item["reason"]
    for item in slm_excluded_retrieval
)

exclusion_summary_df = pd.DataFrame({
    "reason": sorted(
        set(recursive_exclusion_summary)
        | set(slm_exclusion_summary)
    )
})

exclusion_summary_df[
    "recursive_count"
] = exclusion_summary_df[
    "reason"
].map(
    recursive_exclusion_summary
).fillna(0).astype(int)

exclusion_summary_df[
    "slm_count"
] = exclusion_summary_df[
    "reason"
].map(
    slm_exclusion_summary
).fillna(0).astype(int)

display(exclusion_summary_df)

,reason,recursive_count,slm_count
0,metadata_or_footer_noise,1,1
1,navigation_or_admin,8,9
2,reference_heavy,4,1
3,too_short_to_retrieve,218,16


Then inspect the actual excluded chunks:

### Code 089 — Then inspect the actual excluded chunks:


In [ ]:
recursive_excluded_df = pd.DataFrame(
    recursive_excluded_retrieval
)

slm_excluded_df = pd.DataFrame(
    slm_excluded_retrieval
)

print("RECURSIVE EXCLUSIONS")
display(
    recursive_excluded_df[
        [
            "chunk_id",
            "document",
            "page",
            "reason",
            "text"
        ]
    ]
)

print("\nSLM EXCLUSIONS")
display(
    slm_excluded_df[
        [
            "chunk_id",
            "document",
            "page",
            "reason",
            "text"
        ]
    ]
)

RECURSIVE EXCLUSIONS


,chunk_id,document,page,reason,text
0,REC_CHUNK_00016,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,6,navigation_or_admin,National Guidance\n\nESRE 2018 NICE 2018 RCOG\...
1,REC_CHUNK_00054,PCOS-Guideline-Summary-2023-2.pdf,8,too_short_to_retrieve,OOO⊕\n\n+\n\nOOO
2,REC_CHUNK_00061,PCOS-Guideline-Summary-2023-2.pdf,9,too_short_to_retrieve,⊕〇〇O
3,REC_CHUNK_00062,PCOS-Guideline-Summary-2023-2.pdf,9,too_short_to_retrieve,|----------------|--------|-------------------...
4,REC_CHUNK_00067,PCOS-Guideline-Summary-2023-2.pdf,10,too_short_to_retrieve,|----------------|--------|-------------------...
...,...,...,...,...,...
226,REC_CHUNK_01808,What are the treatments for PCOS_ _ NICHD - Eu...,2,reference_heavy,"1. Teede, H. J., Tay, C. T., Laven, J. J. E., ..."
227,REC_CHUNK_01809,What are the treatments for PCOS_ _ NICHD - Eu...,2,reference_heavy,"4. National Heart, Lung, and Blood Institute. ..."
228,REC_CHUNK_01810,What are the treatments for PCOS_ _ NICHD - Eu...,2,reference_heavy,"7. Azarbayjani, K., Jahanian Sadatmahalleh, S...."
229,REC_CHUNK_01813,What are the treatments for PCOS_ _ NICHD - Eu...,3,reference_heavy,13. Radosh L. (2009). Drug treatments for poly...



SLM EXCLUSIONS


,chunk_id,document,page,reason,text
0,SLM_CHUNK_00014,PCOS-Gail-Busby-and-Montilla-002 (1).pdf,6,navigation_or_admin,National Guidance ESRE 2018 NICE 2018 RCOG Ref...
1,SLM_CHUNK_00068,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,1,navigation_or_admin,"Although the cause of PCOS is not known, PCOS ..."
2,SLM_CHUNK_00086,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,5,navigation_or_admin,FAQ121 Last updated: January 2022 Last reviewe...
3,SLM_CHUNK_00087,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,5,navigation_or_admin,    About ACOG Disclaimer Contact Us How t...
4,SLM_CHUNK_00097,Polycystic ovary syndrome.pdf,3,too_short_to_retrieve,Possible symptoms include:
5,SLM_CHUNK_00114,Updated-PMOS-Guideline-12-June26.pdf,6,navigation_or_admin,Copyright information © Monash University Pape...
6,SLM_CHUNK_00115,Updated-PMOS-Guideline-12-June26.pdf,6,navigation_or_admin,Electronic documents: This work is subject to ...
7,SLM_CHUNK_00116,Updated-PMOS-Guideline-12-June26.pdf,6,navigation_or_admin,Copies of the guideline and the technical repo...
8,SLM_CHUNK_00119,Updated-PMOS-Guideline-12-June26.pdf,9,too_short_to_retrieve,Chapter Three Chapter Six
9,SLM_CHUNK_00216,Updated-PMOS-Guideline-12-June26.pdf,57,too_short_to_retrieve,O Q O C


If the exclusions look correct, save the final retrieval-ready corpora:

### Code 090 — If the exclusions look correct, save the final retrieval-ready corpora:


In [ ]:
recursive_retrieval_rows = []

for chunk in recursive_retrieval_chunks:
    recursive_retrieval_rows.append({
        "chunk_id":
            chunk.metadata.get("chunk_id"),

        "document_name":
            chunk.metadata.get("document_name"),

        "pdf_page_number":
            chunk.metadata.get("pdf_page_number"),

        "organization":
            chunk.metadata.get("organization"),

        "source_type":
            chunk.metadata.get("source_type"),

        "chunking_method":
            chunk.metadata.get("chunking_method"),

        "word_count":
            len(chunk.page_content.split()),

        "character_count":
            len(chunk.page_content),

        "text":
            chunk.page_content
    })


slm_retrieval_rows = []

for chunk in slm_retrieval_chunks:
    slm_retrieval_rows.append({
        "chunk_id":
            chunk.metadata.get("chunk_id"),

        "document_name":
            chunk.metadata.get("document_name"),

        "pdf_page_number":
            chunk.metadata.get("pdf_page_number"),

        "organization":
            chunk.metadata.get("organization"),

        "source_type":
            chunk.metadata.get("source_type"),

        "chunking_method":
            chunk.metadata.get("chunking_method"),

        "word_count":
            len(chunk.page_content.split()),

        "character_count":
            len(chunk.page_content),

        "text":
            chunk.page_content
    })


recursive_retrieval_df = pd.DataFrame(
    recursive_retrieval_rows
)

slm_retrieval_df = pd.DataFrame(
    slm_retrieval_rows
)

### Code 091 — Export Results


In [ ]:
recursive_retrieval_df.to_csv(
    os.path.join(
        CHUNKS_FOLDER,
        "recursive_retrieval_ready.csv"
    ),
    index=False
)

slm_retrieval_df.to_csv(
    os.path.join(
        CHUNKS_FOLDER,
        "slm_retrieval_ready.csv"
    ),
    index=False
)

print("Retrieval-ready chunk sets saved.")

Retrieval-ready chunk sets saved.


#Build Retrieval Evaluation Dataset

### Create the Retrieval Benchmark Question Set

The 20 questions below are a researcher-defined benchmark covering major PCOS support topics. They are not called a "gold" question set unless independent source-level qrels are later established.

### Code 092 — Create the Retrieval Benchmark Question Set


In [ ]:
retrieval_test_questions = [

    {
        "question_id": "Q01",
        "question": "What lifestyle changes are recommended for women with PCOS?",
        "category": "general_lifestyle"
    },

    {
        "question_id": "Q02",
        "question": "What physical activity is recommended for women with PCOS?",
        "category": "physical_activity"
    },

    {
        "question_id": "Q03",
        "question": "How can weight management support women with PCOS?",
        "category": "weight_management"
    },

    {
        "question_id": "Q04",
        "question": "Is there one specific diet that is recommended for PCOS?",
        "category": "nutrition"
    },

    {
        "question_id": "Q05",
        "question": "What healthy eating advice is recommended for PCOS?",
        "category": "nutrition"
    },

    {
        "question_id": "Q06",
        "question": "What menstrual cycle patterns are associated with PCOS?",
        "category": "menstrual_awareness"
    },

    {
        "question_id": "Q07",
        "question": "What is the relationship between PCOS and excess hair growth?",
        "category": "hyperandrogenism_awareness"
    },

    {
        "question_id": "Q08",
        "question": "How is acne related to PCOS?",
        "category": "dermatological_awareness"
    },

    {
        "question_id": "Q09",
        "question": "What metabolic health risks are associated with PCOS?",
        "category": "metabolic_awareness"
    },

    {
        "question_id": "Q10",
        "question": "How is insulin resistance related to PCOS?",
        "category": "metabolic_awareness"
    },

    {
        "question_id": "Q11",
        "question": "What psychological problems can affect women with PCOS?",
        "category": "emotional_wellbeing"
    },

    {
        "question_id": "Q12",
        "question": "When should psychological support be considered for women with PCOS?",
        "category": "professional_support"
    },

    {
        "question_id": "Q13",
        "question": "Can lifestyle management provide benefits even without weight loss in PCOS?",
        "category": "weight_management"
    },

    {
        "question_id": "Q14",
        "question": "What are the benefits of regular physical activity for PCOS?",
        "category": "physical_activity"
    },

    {
        "question_id": "Q15",
        "question": "How can healthcare professionals support sustainable lifestyle changes in PCOS?",
        "category": "general_lifestyle"
    },

    {
        "question_id": "Q16",
        "question": "What reproductive health issues are associated with PCOS?",
        "category": "menstrual_awareness"
    },

    {
        "question_id": "Q17",
        "question": "Why is prevention of excess weight gain important in PCOS?",
        "category": "weight_management"
    },

    {
        "question_id": "Q18",
        "question": "What role does sleep have in the health of women with PCOS?",
        "category": "general_lifestyle"
    },

    {
        "question_id": "Q19",
        "question": "How can PCOS affect quality of life?",
        "category": "emotional_wellbeing"
    },

    {
        "question_id": "Q20",
        "question": "What should women with PCOS know about long-term health risks?",
        "category": "general_pcos"
    }
]

Cell 2 — Convert and save the question set

### Code 093 — Cell 2 — Convert and save the question set


In [ ]:
retrieval_questions_df = pd.DataFrame(
    retrieval_test_questions
)

display(
    retrieval_questions_df
)

,question_id,question,category
0,Q01,What lifestyle changes are recommended for wom...,general_lifestyle
1,Q02,What physical activity is recommended for wome...,physical_activity
2,Q03,How can weight management support women with P...,weight_management
3,Q04,Is there one specific diet that is recommended...,nutrition
4,Q05,What healthy eating advice is recommended for ...,nutrition
5,Q06,What menstrual cycle patterns are associated w...,menstrual_awareness
6,Q07,What is the relationship between PCOS and exce...,hyperandrogenism_awareness
7,Q08,How is acne related to PCOS?,dermatological_awareness
8,Q09,What metabolic health risks are associated wit...,metabolic_awareness
9,Q10,How is insulin resistance related to PCOS?,metabolic_awareness


### Code 094 — Prepare retrieval_questions_path


In [ ]:
retrieval_questions_path = os.path.join(
    EVALUATION_FOLDER,
    "retrieval_test_questions.csv"
)

retrieval_questions_df.to_csv(
    retrieval_questions_path,
    index=False
)

print(
    "Saved:",
    retrieval_questions_path
)

Saved: /content/drive/MyDrive/pcos_rag/evaluation/retrieval_test_questions.csv


## Historical Dense-Retrieval Experiment — FAISS Retained for Reproducibility

The following FAISS cells are part of the original controlled research experiments and are intentionally retained so those results remain reproducible.

**They are not the final operational vector-store implementation.**

The final deployed pipeline keeps the same MiniLM embedding model and cosine-similarity semantics, but the final retrievable corpus is stored/searched in **Qdrant**. The Qdrant-ready handoff is defined at the end of this notebook.


same dense retriever for both Recursive and SLM-assisted chunk sets

Step 1 — Install and import

### Code 095 — Step 1 — Install and import


In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 126.2 MB/s eta 0:00:00


### Code 096 — Imports and Setup


In [ ]:
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer

Step 2 — Record retrieval configuration

### Code 097 — Step 2 — Record retrieval configuration


In [ ]:
DENSE_RETRIEVAL_CONFIG = {

    "embedding_model":
        "sentence-transformers/all-MiniLM-L6-v2",

    "similarity":
        "cosine",

    "normalize_embeddings":
        True,

    "faiss_index":
        "IndexFlatIP",

    # Evaluation horizon - not claimed as universally optimal
    "candidate_top_k":
        10,

    # We will report performance at multiple K values
    "evaluation_k_values":
        [1, 3, 5, 10],

    "batch_size":
        64
}

DENSE_RETRIEVAL_CONFIG

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'similarity': 'cosine',
 'normalize_embeddings': True,
 'faiss_index': 'IndexFlatIP',
 'candidate_top_k': 10,
 'evaluation_k_values': [1, 3, 5, 10],
 'batch_size': 64}

Important distinction:

embedding_model, similarity method and FAISS index are methodological choices.
top_k=10 is currently an evaluation setting, not a claimed optimal hyperparameter.
We evaluate K = 1,3,5,10 rather than deciding one K is best before seeing the results.
batch_size=64 affects processing efficiency, not retrieval semantics.

Save the config:

### Code 098 — Save the config:


In [ ]:
dense_config_df = pd.DataFrame(
    [
        {
            "parameter": key,
            "value": str(value)
        }
        for key, value
        in DENSE_RETRIEVAL_CONFIG.items()
    ]
)

display(dense_config_df)

dense_config_df.to_csv(
    os.path.join(
        EVALUATION_FOLDER,
        "dense_retrieval_configuration.csv"
    ),
    index=False
)

print("Dense retrieval configuration saved.")

,parameter,value
0,embedding_model,sentence-transformers/all-MiniLM-L6-v2
1,similarity,cosine
2,normalize_embeddings,True
3,faiss_index,IndexFlatIP
4,candidate_top_k,10
5,evaluation_k_values,"[1, 3, 5, 10]"
6,batch_size,64


Dense retrieval configuration saved.


Step 3 — Load the embedding model

### Code 099 — Step 3 — Load the embedding model


In [ ]:
EMBEDDING_MODEL_NAME = (
    DENSE_RETRIEVAL_CONFIG[
        "embedding_model"
    ]
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME
)

print(
    "Device:",
    embedding_model.device
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda:0


Step 4 — Prepare both retrieval corpora

Recursive:

### Code 100 — Recursive:


In [ ]:
recursive_texts = [
    chunk.page_content
    for chunk
    in recursive_retrieval_chunks
]

print(
    "Recursive texts:",
    len(recursive_texts)
)

Recursive texts: 1584


SLM:

### Code 101 — SLM:


In [ ]:
slm_texts = [
    chunk.page_content
    for chunk
    in slm_retrieval_chunks
]

print(
    "SLM texts:",
    len(slm_texts)
)

SLM texts: 932


Step 5 — Generate Recursive embeddings

### Code 102 — Step 5 — Generate Recursive embeddings


In [ ]:
recursive_embeddings = embedding_model.encode(

    recursive_texts,

    batch_size=64,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

recursive_embeddings = (
    recursive_embeddings.astype(
        "float32"
    )
)

print(
    "Recursive embedding shape:",
    recursive_embeddings.shape
)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Recursive embedding shape: (1584, 384)


Step 6 — Generate SLM embeddings

### Code 103 — Step 6 — Generate SLM embeddings


In [ ]:
slm_embeddings = embedding_model.encode(

    slm_texts,

    batch_size=64,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

slm_embeddings = (
    slm_embeddings.astype(
        "float32"
    )
)

print(
    "SLM embedding shape:",
    slm_embeddings.shape
)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

SLM embedding shape: (932, 384)


Step 7 — Verify normalization

### Code 104 — Step 7 — Verify normalization


In [ ]:
recursive_norms = np.linalg.norm(
    recursive_embeddings,
    axis=1
)

slm_norms = np.linalg.norm(
    slm_embeddings,
    axis=1
)

print(
    "Recursive average norm:",
    recursive_norms.mean()
)

print(
    "SLM average norm:",
    slm_norms.mean()
)

Recursive average norm: 1.0
SLM average norm: 1.0


Step 8 — Build exact FAISS indexes

Recursive:

### Code 105 — Recursive:


In [ ]:
embedding_dimension = (
    recursive_embeddings.shape[1]
)

recursive_faiss_index = (
    faiss.IndexFlatIP(
        embedding_dimension
    )
)

recursive_faiss_index.add(
    recursive_embeddings
)

print(
    "Recursive FAISS vectors:",
    recursive_faiss_index.ntotal
)

Recursive FAISS vectors: 1584


SLM:

### Code 106 — SLM:


In [ ]:
slm_faiss_index = (
    faiss.IndexFlatIP(
        embedding_dimension
    )
)

slm_faiss_index.add(
    slm_embeddings
)

print(
    "SLM FAISS vectors:",
    slm_faiss_index.ntotal
)

SLM FAISS vectors: 932


Step 9 — Common retrieval function

Both chunking methods must use exactly the same function.

### Code 107 — Step 9 — Common retrieval function


In [ ]:
def dense_retrieve(
    query,
    index,
    chunks,
    embedding_model,
    top_k=10
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(
            scores[0],
            indices[0]
        ),
        start=1
    ):

        if idx < 0:
            continue

        chunk = chunks[idx]

        results.append({

            "rank":
                rank,

            "chunk_id":
                chunk.metadata.get(
                    "chunk_id"
                ),

            "document_name":
                chunk.metadata.get(
                    "document_name"
                ),

            "pdf_page_number":
                chunk.metadata.get(
                    "pdf_page_number"
                ),

            "organization":
                chunk.metadata.get(
                    "organization"
                ),

            "similarity_score":
                float(score),

            "text":
                chunk.page_content
        })

    return results

Step 10 — Test one question first

### Code 108 — Step 10 — Test one question first


In [ ]:
test_question = (
    retrieval_test_questions[0][
        "question"
    ]
)

print(
    "TEST QUESTION:"
)

print(
    test_question
)

TEST QUESTION:
What lifestyle changes are recommended for women with PCOS?


Recursive:

### Code 109 — Recursive:


In [ ]:
recursive_test_results = dense_retrieve(

    query=
        test_question,

    index=
        recursive_faiss_index,

    chunks=
        recursive_retrieval_chunks,

    embedding_model=
        embedding_model,

    top_k=10
)

recursive_test_df = pd.DataFrame(
    recursive_test_results
)

display(
    recursive_test_df[
        [
            "rank",
            "chunk_id",
            "document_name",
            "pdf_page_number",
            "similarity_score",
            "text"
        ]
    ]
)

,rank,chunk_id,document_name,pdf_page_number,similarity_score,text
0,1,REC_CHUNK_00241,Polycystic ovary syndrome.pdf,4,0.735077,"- intermittent, unpredictable or absent period..."
1,2,REC_CHUNK_00120,PCOS-Guideline-Summary-2023-2.pdf,19,0.729402,| 3 | | Lifestyle management | |\n| 3.1 | | Ef...
2,3,REC_CHUNK_00245,Polycystic ovary syndrome.pdf,5,0.728030,Women who are newly diagnosed with PCOS are re...
3,4,REC_CHUNK_01687,Updated-PMOS-Guideline-12-June26.pdf,244,0.724863,Lifestyle interventions could include behaviou...
4,5,REC_CHUNK_00227,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,0.724100,What effect can weight loss have on women with...
5,6,REC_CHUNK_00141,PCOS-Guideline-Summary-2023-2.pdf,22,0.721414,| 3.5 | | Factors affecting weight gain in PCO...
6,7,REC_CHUNK_00239,Polycystic ovary syndrome.pdf,3,0.719327,"Further, PCOS is a chronic metabolic condition..."
7,8,REC_CHUNK_00865,Updated-PMOS-Guideline-12-June26.pdf,103,0.710591,| 3.1.1 | EBR | Lifestyle intervention (exerci...
8,9,REC_CHUNK_00465,Updated-PMOS-Guideline-12-June26.pdf,42,0.708935,| 3.5 | | Factors affecting weight gain in PMO...
9,10,REC_CHUNK_00121,PCOS-Guideline-Summary-2023-2.pdf,19,0.703157,| 3.1.3 | PP | Healthcare professionals should...


SLM:

### Code 110 — SLM:


In [ ]:
slm_test_results = dense_retrieve(

    query=
        test_question,

    index=
        slm_faiss_index,

    chunks=
        slm_retrieval_chunks,

    embedding_model=
        embedding_model,

    top_k=10
)

slm_test_df = pd.DataFrame(
    slm_test_results
)

display(
    slm_test_df[
        [
            "rank",
            "chunk_id",
            "document_name",
            "pdf_page_number",
            "similarity_score",
            "text"
        ]
    ]
)

,rank,chunk_id,document_name,pdf_page_number,similarity_score,text
0,1,SLM_CHUNK_00770,Updated-PMOS-Guideline-12-June26.pdf,244,0.755900,Dietary intervention General healthy eating pr...
1,2,SLM_CHUNK_00098,Polycystic ovary syndrome.pdf,4,0.735077,"- intermittent, unpredictable or absent period..."
2,3,SLM_CHUNK_00766,Updated-PMOS-Guideline-12-June26.pdf,243,0.732159,Effectiveness of lifestyle interventions Healt...
3,4,SLM_CHUNK_00102,Polycystic ovary syndrome.pdf,5,0.728030,Women who are newly diagnosed with PCOS are re...
4,5,SLM_CHUNK_00769,Updated-PMOS-Guideline-12-June26.pdf,244,0.720706,Lifestyle interventions could include behaviou...
5,6,SLM_CHUNK_00357,Updated-PMOS-Guideline-12-June26.pdf,102,0.705596,3.1 Effectiveness of lifestyle interventions I...
6,7,SLM_CHUNK_00628,Updated-PMOS-Guideline-12-June26.pdf,179,0.704804,- How can we best support women to navigate th...
7,8,SLM_CHUNK_00631,Updated-PMOS-Guideline-12-June26.pdf,179,0.703764,"- In women with PMOS/PCOS, are diet interventi..."
8,9,SLM_CHUNK_00090,Polycystic ovary syndrome.pdf,1,0.695954,- PCOS runs in families but can cause a range ...
9,10,SLM_CHUNK_00075,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,0.691629,What effect can weight loss have on women with...


Step 11 — Run all 20 questions

### Code 111 — Step 11 — Run all 20 questions


In [ ]:
all_retrieval_results = []

TOP_K = 10

for item in retrieval_test_questions:

    question_id = item["question_id"]
    question = item["question"]
    category = item["category"]

    # -------------------------
    # Recursive retrieval
    # -------------------------

    recursive_results = dense_retrieve(
        query=question,
        index=recursive_faiss_index,
        chunks=recursive_retrieval_chunks,
        embedding_model=embedding_model,
        top_k=TOP_K
    )

    for result in recursive_results:

        all_retrieval_results.append({

            "question_id":
                question_id,

            "question":
                question,

            "category":
                category,

            "chunking_method":
                "recursive",

            **result
        })


    # -------------------------
    # SLM retrieval
    # -------------------------

    slm_results = dense_retrieve(
        query=question,
        index=slm_faiss_index,
        chunks=slm_retrieval_chunks,
        embedding_model=embedding_model,
        top_k=TOP_K
    )

    for result in slm_results:

        all_retrieval_results.append({

            "question_id":
                question_id,

            "question":
                question,

            "category":
                category,

            "chunking_method":
                "slm_assisted",

            **result
        })


print(
    "Total retrieved rows:",
    len(all_retrieval_results)
)

Total retrieved rows: 400


Step 12 — Convert to DataFrame

### Code 112 — Step 12 — Convert to DataFrame


In [ ]:
retrieval_results_df = pd.DataFrame(
    all_retrieval_results
)

print(
    "Rows:",
    len(retrieval_results_df)
)

display(
    retrieval_results_df.head(20)
)

Rows: 400


,question_id,question,category,chunking_method,rank,chunk_id,document_name,pdf_page_number,organization,similarity_score,text
0,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,1,REC_CHUNK_00241,Polycystic ovary syndrome.pdf,4,World Health Organization,0.735077,"- intermittent, unpredictable or absent period..."
1,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,2,REC_CHUNK_00120,PCOS-Guideline-Summary-2023-2.pdf,19,International PCOS Guideline,0.729402,| 3 | | Lifestyle management | |\n| 3.1 | | Ef...
2,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,3,REC_CHUNK_00245,Polycystic ovary syndrome.pdf,5,World Health Organization,0.728030,Women who are newly diagnosed with PCOS are re...
3,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,4,REC_CHUNK_01687,Updated-PMOS-Guideline-12-June26.pdf,244,International PCOS Guideline,0.724863,Lifestyle interventions could include behaviou...
4,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,5,REC_CHUNK_00227,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,unknown,0.724100,What effect can weight loss have on women with...
5,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,6,REC_CHUNK_00141,PCOS-Guideline-Summary-2023-2.pdf,22,International PCOS Guideline,0.721414,| 3.5 | | Factors affecting weight gain in PCO...
6,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,7,REC_CHUNK_00239,Polycystic ovary syndrome.pdf,3,World Health Organization,0.719327,"Further, PCOS is a chronic metabolic condition..."
7,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,8,REC_CHUNK_00865,Updated-PMOS-Guideline-12-June26.pdf,103,International PCOS Guideline,0.710591,| 3.1.1 | EBR | Lifestyle intervention (exerci...
8,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,9,REC_CHUNK_00465,Updated-PMOS-Guideline-12-June26.pdf,42,International PCOS Guideline,0.708935,| 3.5 | | Factors affecting weight gain in PMO...
9,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,10,REC_CHUNK_00121,PCOS-Guideline-Summary-2023-2.pdf,19,International PCOS Guideline,0.703157,| 3.1.3 | PP | Healthcare professionals should...


Step 13 — Check counts

### Code 113 — Step 13 — Check counts


In [ ]:
retrieval_count_check = (
    retrieval_results_df
    .groupby(
        [
            "question_id",
            "chunking_method"
        ]
    )
    .size()
    .reset_index(
        name="retrieved_count"
    )
)

display(
    retrieval_count_check
)

,question_id,chunking_method,retrieved_count
0,Q01,recursive,10
1,Q01,slm_assisted,10
2,Q02,recursive,10
3,Q02,slm_assisted,10
4,Q03,recursive,10
5,Q03,slm_assisted,10
6,Q04,recursive,10
7,Q04,slm_assisted,10
8,Q05,recursive,10
9,Q05,slm_assisted,10


Step 14 — Save raw retrieval results

### Code 114 — Step 14 — Save raw retrieval results


In [ ]:
raw_results_path = os.path.join(
    EVALUATION_FOLDER,
    "dense_retrieval_all_results.csv"
)

retrieval_results_df.to_csv(
    raw_results_path,
    index=False
)

print(
    "Saved:",
    raw_results_path
)

Saved: /content/drive/MyDrive/pcos_rag/evaluation/dense_retrieval_all_results.csv


## AI-Assisted Graded Relevance Annotation

For each retrieved chunk:

- **2 = Highly relevant**
- **1 = Partially relevant**
- **0 = Irrelevant**

These are **AI-assisted reference annotations** (e.g., ChatGPT applying the predefined rubric) unless independently human-validated. Do not present them as expert human gold-standard judgments.

For binary Precision/MRR/Recall, labels 1 and 2 are treated as relevant. Graded 0/1/2 labels are retained for nDCG.


### Code 115 — AI-Assisted Graded Relevance Annotation


In [ ]:
import pandas as pd
import os
from google.colab import files

# Define the paths
labeled_filename = "dense_retrieval_all_results_labeled.csv"
labeled_path_in_drive = os.path.join(
    EVALUATION_FOLDER, labeled_filename
)

# Initialize retrieval_label_df to None. This will be updated after upload.
retrieval_label_df = None

# Always proceed with generating a template and prompting for upload.
# This ensures that only an uploaded CSV is used, as requested.
print(
    "Generating a template for manual labeling. "
    "Please download, label, and upload the file."
)
# retrieval_results_df is available from previous cells.

# Create a copy to add relevance columns
retrieval_label_template_df = retrieval_results_df.copy()
retrieval_label_template_df["relevance_label"] = ""
retrieval_label_template_df["relevance_notes"] = ""
retrieval_label_template_df["annotation_method"] = (
    "AI-assisted graded relevance rubric; researcher/human validation recommended"
)

temp_template_path = "dense_retrieval_all_results_for_labeling.csv"
retrieval_label_template_df.to_csv(temp_template_path, index=False)
print(f"Please download '{temp_template_path}', add your relevance labels, and then upload it back.")
print(f"After labeling, please rename the file to '{labeled_filename}' before uploading.")

# Offer download functionality
files.download(temp_template_path)

print(f"\nUpload the labeled file '{labeled_filename}' to continue:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded. Please upload the labeled CSV to continue.")

uploaded_filename = list(uploaded.keys())[0]

# Ensure the uploaded file is saved to the expected path in Google Drive
# regardless of its initial name, if it's a CSV.
if uploaded_filename != labeled_filename:
    print(
        f"Warning: Uploaded file name '{uploaded_filename}' does not match "
        f"expected file name '{labeled_filename}'. "
        "Saving the uploaded file with the expected name."
    )
    # Save the uploaded file with the correct expected name
    with open(labeled_path_in_drive, "wb") as f:
        f.write(uploaded[uploaded_filename])
    print(f"Successfully uploaded and saved '{uploaded_filename}' as '{labeled_path_in_drive}'.")
else:
    # Save the uploaded file to the expected path in EVALUATION_FOLDER
    with open(labeled_path_in_drive, "wb") as f:
        f.write(uploaded[uploaded_filename])
    print(f"Successfully uploaded and saved '{uploaded_filename}' to '{labeled_path_in_drive}'.")

retrieval_label_df = pd.read_csv(labeled_path_in_drive)


print("Rows:", len(retrieval_label_df))
display(retrieval_label_df.head())

Generating a template for manual labeling. Please download, label, and upload the file.
Please download 'dense_retrieval_all_results_for_labeling.csv', add your relevance labels, and then upload it back.
After labeling, please rename the file to 'dense_retrieval_all_results_labeled.csv' before uploading.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Upload the labeled file 'dense_retrieval_all_results_labeled.csv' to continue:


Saving dense_retrieval_all_results_labeled.csv to dense_retrieval_all_results_labeled.csv
Successfully uploaded and saved 'dense_retrieval_all_results_labeled.csv' to '/content/drive/MyDrive/pcos_rag/evaluation/dense_retrieval_all_results_labeled.csv'.
Rows: 400


,question_id,question,category,chunking_method,rank,chunk_id,document_name,pdf_page_number,organization,similarity_score,text,graded_relevance,relevance_label,binary_relevant,annotation_type,gold_standard_status
0,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,1,REC_CHUNK_00241,Polycystic ovary syndrome.pdf,4,World Health Organization,0.735077,"- intermittent, unpredictable or absent period...",0,Irrelevant,0,AI-assisted reference annotation,Not independently human-validated
1,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,2,REC_CHUNK_00120,PCOS-Guideline-Summary-2023-2.pdf,19,International PCOS Guideline,0.729402,| 3 | | Lifestyle management | |\n| 3.1 | | Ef...,2,Highly relevant,1,AI-assisted reference annotation,Not independently human-validated
2,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,3,REC_CHUNK_00245,Polycystic ovary syndrome.pdf,5,World Health Organization,0.728030,Women who are newly diagnosed with PCOS are re...,0,Irrelevant,0,AI-assisted reference annotation,Not independently human-validated
3,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,4,REC_CHUNK_01687,Updated-PMOS-Guideline-12-June26.pdf,244,International PCOS Guideline,0.724863,Lifestyle interventions could include behaviou...,2,Highly relevant,1,AI-assisted reference annotation,Not independently human-validated
4,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,5,REC_CHUNK_00227,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,unknown,0.724100,What effect can weight loss have on women with...,1,Partially relevant,1,AI-assisted reference annotation,Not independently human-validated


### Code 116 — Prepare retrieval_label_df


In [ ]:
retrieval_label_df = retrieval_label_df.copy()

retrieval_label_df["relevance_label"] = ""
retrieval_label_df["relevance_notes"] = ""
retrieval_label_df["annotation_method"] = (
    "AI-assisted graded relevance rubric; researcher/human validation recommended"
)


Create a blind evaluation ID. Hiding retrieval method, similarity score, and rank reduces method-related annotation bias.

### Code 117 — Notebook Processing Step


In [ ]:
retrieval_label_df[
    "evaluation_id"
] = [
    f"RET_{i:04d}"
    for i in range(
        len(retrieval_label_df)
    )
]

### Code 118 — Prepare retrieval_label_df


In [ ]:
retrieval_label_df = (
    retrieval_label_df
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(
        drop=True
    )
)

Create the blind annotation CSV. The question remains visible; the retrieval method, similarity score, and rank remain hidden.

### Code 119 — Define blind_columns


In [ ]:
blind_columns = [

    "evaluation_id",

    "question_id",

    "question",

    "category",

    "document_name",

    "pdf_page_number",

    "text",

    "relevance_label",

    "relevance_notes"
]

blind_relevance_df = (
    retrieval_label_df[
        blind_columns
    ].copy()
)

display(
    blind_relevance_df.head(10)
)

,evaluation_id,question_id,question,category,document_name,pdf_page_number,text,relevance_label,relevance_notes
0,RET_0209,Q10,How is insulin resistance related to PCOS?,metabolic_awareness,Updated-PMOS-Guideline-12-June26.pdf,103,| 3.1.7 | PP | Healthcare professionals should...,,
1,RET_0280,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,What effect can weight loss have on women with...,,
2,RET_0033,Q03,How can weight management support women with P...,weight_management,Updated-PMOS-Guideline-12-June26.pdf,111,Intrinsic factors potentially contributing to ...,,
3,RET_0210,Q12,When should psychological support be considere...,professional_support,PCOS-Guideline-Summary-2023-2.pdf,18,| 2.7 | | Psychological therapy | |\n| 2.7.1 |...,,
4,RET_0093,Q12,When should psychological support be considere...,professional_support,PCOS-Guideline-Summary-2023-2.pdf,4,The value and optimal timing of assessment and...,,
5,RET_0084,Q14,What are the benefits of regular physical acti...,physical_activity,Updated-PMOS-Guideline-12-June26.pdf,244,Lifestyle interventions could include behaviou...,,
6,RET_0329,Q09,What metabolic health risks are associated wit...,metabolic_awareness,NICHDWHPCOSweb.pdf,1,What is PCOS?\n\nPCOS describes a group of sym...,,
7,RET_0094,Q10,How is insulin resistance related to PCOS?,metabolic_awareness,Updated-PMOS-Guideline-12-June26.pdf,77,| 1.9.12 | PP | Insulin resistance is a pathop...,,
8,RET_0266,Q16,What reproductive health issues are associated...,menstrual_awareness,Polycystic ovary syndrome.pdf,2,variety of longer-term health problems that af...,,
9,RET_0126,Q18,What role does sleep have in the health of wom...,general_lifestyle,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,2,Women with PCOS also may be at higher risk of ...,,


### Code 120 — Prepare blind_label_path


In [ ]:
blind_label_path = os.path.join(
    EVALUATION_FOLDER,
    "retrieval_relevance_blind_400.csv"
)

blind_relevance_df.to_csv(
    blind_label_path,
    index=False
)

print(
    "Saved blind labeling file:",
    blind_label_path
)

Saved blind labeling file: /content/drive/MyDrive/pcos_rag/evaluation/retrieval_relevance_blind_400.csv


Step 18 — Save the hidden mapping separately

We still need the original method/rank later.

### Code 121 — Step 18 — Save the hidden mapping separately


In [ ]:
mapping_columns = [

    "evaluation_id",

    "question_id",

    "chunking_method",

    "rank",

    "chunk_id",

    "similarity_score"
]

retrieval_mapping_df = (
    retrieval_label_df[
        mapping_columns
    ].copy()
)

mapping_path = os.path.join(
    EVALUATION_FOLDER,
    "retrieval_relevance_hidden_mapping.csv"
)

retrieval_mapping_df.to_csv(
    mapping_path,
    index=False
)

print(
    "Saved hidden mapping:",
    mapping_path
)

Saved hidden mapping: /content/drive/MyDrive/pcos_rag/evaluation/retrieval_relevance_hidden_mapping.csv


### Code 122 — Imports and Setup


In [ ]:
import pandas as pd

labeled_path = os.path.join(
    EVALUATION_FOLDER,
    "dense_retrieval_all_results_labeled.csv"
)

labeled_df = pd.read_csv(labeled_path)

print("Rows:", len(labeled_df))
print(
    "Annotation provenance: AI-assisted graded-relevance reference labels "
    "unless separately human-validated."
)
display(labeled_df.head())


Rows: 400
Annotation provenance: AI-assisted graded-relevance reference labels unless separately human-validated.


,question_id,question,category,chunking_method,rank,chunk_id,document_name,pdf_page_number,organization,similarity_score,text,graded_relevance,relevance_label,binary_relevant,annotation_type,gold_standard_status
0,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,1,REC_CHUNK_00241,Polycystic ovary syndrome.pdf,4,World Health Organization,0.735077,"- intermittent, unpredictable or absent period...",0,Irrelevant,0,AI-assisted reference annotation,Not independently human-validated
1,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,2,REC_CHUNK_00120,PCOS-Guideline-Summary-2023-2.pdf,19,International PCOS Guideline,0.729402,| 3 | | Lifestyle management | |\n| 3.1 | | Ef...,2,Highly relevant,1,AI-assisted reference annotation,Not independently human-validated
2,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,3,REC_CHUNK_00245,Polycystic ovary syndrome.pdf,5,World Health Organization,0.728030,Women who are newly diagnosed with PCOS are re...,0,Irrelevant,0,AI-assisted reference annotation,Not independently human-validated
3,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,4,REC_CHUNK_01687,Updated-PMOS-Guideline-12-June26.pdf,244,International PCOS Guideline,0.724863,Lifestyle interventions could include behaviou...,2,Highly relevant,1,AI-assisted reference annotation,Not independently human-validated
4,Q01,What lifestyle changes are recommended for wom...,general_lifestyle,recursive,5,REC_CHUNK_00227,Polycystic Ovary Syndrome (PCOS) _ ACOG.pdf,3,unknown,0.724100,What effect can weight loss have on women with...,1,Partially relevant,1,AI-assisted reference annotation,Not independently human-validated


Convert the AI-assisted graded relevance column to numeric.

### Code 123 — Convert the AI-assisted graded relevance column to numeric.


In [ ]:


print(
    labeled_df["relevance_label"]
    .value_counts(dropna=False)
    .sort_index()
)

relevance_label
Highly relevant       201
Irrelevant             58
Partially relevant    141
Name: count, dtype: int64


Check missing labels:

### Code 124 — Check missing labels:


In [ ]:
missing_labels = labeled_df[
    labeled_df["relevance_label"].isna()
]

print("Missing labels:", len(missing_labels))

Missing labels: 0


### Code 126 — Then create the binary relevance column:


In [ ]:
relevance_mapping = {
    "Irrelevant": 0,
    "Partially relevant": 1,
    "Highly relevant": 2
}

labeled_df["relevance_label"] = labeled_df["relevance_label"].map(relevance_mapping).fillna(labeled_df["relevance_label"])
labeled_df["relevance_label"] = pd.to_numeric(labeled_df["relevance_label"], errors="coerce")

labeled_df["is_relevant"] = (
    labeled_df["relevance_label"] >= 1
).astype(int)

### Code 127 — Display Results


In [ ]:
display(
    labeled_df[
        [
            "question_id",
            "chunking_method",
            "rank",
            "relevance_label",
            "is_relevant"
        ]
    ].head(20)
)

,question_id,chunking_method,rank,relevance_label,is_relevant
0,Q01,recursive,1,0,0
1,Q01,recursive,2,2,1
2,Q01,recursive,3,0,0
3,Q01,recursive,4,2,1
4,Q01,recursive,5,1,1
5,Q01,recursive,6,1,1
6,Q01,recursive,7,1,1
7,Q01,recursive,8,2,1
8,Q01,recursive,9,1,1
9,Q01,recursive,10,1,1


## Precision@K

Binary relevance is defined as `relevance_label >= 1`.

\[
P@K = \frac{\#\text{ relevant retrieved items in top }K}{K}
\]

Precision@K is standard; its validity still depends on the quality of the relevance annotations.


### Code 128 — Precision@K


In [ ]:
def precision_at_k(group, k):

    top_k = (
        group
        .sort_values("rank")
        .head(k)
    )

    return top_k["is_relevant"].mean()

### Code 129 — Define K_VALUES


In [ ]:
K_VALUES = [1, 3, 5, 10]

precision_rows = []

for method in labeled_df["chunking_method"].unique():

    method_df = labeled_df[
        labeled_df["chunking_method"] == method
    ]

    for k in K_VALUES:

        question_scores = []

        for question_id, group in method_df.groupby("question_id"):

            score = precision_at_k(
                group,
                k
            )

            question_scores.append(score)

        precision_rows.append({
            "chunking_method": method,
            "k": k,
            "precision": sum(question_scores) / len(question_scores)
        })

### Code 130 — Prepare precision_df


In [ ]:
precision_df = pd.DataFrame(
    precision_rows
)

display(
    precision_df.round(3)
)

,chunking_method,k,precision
0,recursive,1,0.950
1,recursive,3,0.917
2,recursive,5,0.920
3,recursive,10,0.900
4,slm_assisted,1,0.950
5,slm_assisted,3,0.917
6,slm_assisted,5,0.840
7,slm_assisted,10,0.810


## MRR@10

Only the top 10 candidates are judged, so reciprocal rank is reported at that horizon:

\[
RR_q =
\begin{cases}
1/r_q, & \text{if a relevant result occurs in the top 10}\\
0, & \text{otherwise}
\end{cases}
\]

\[
MRR@10 = \frac{1}{Q}\sum_{q=1}^{Q}RR_q
\]


### Code 131 — MRR@10


In [ ]:
def reciprocal_rank_at_k(group, k=TOP_K):

    group = group.sort_values("rank").head(k)
    relevant = group[group["is_relevant"] == 1]

    if len(relevant) == 0:
        return 0.0

    return 1.0 / relevant["rank"].min()


### Code 132 — Define mrr_rows


In [ ]:
mrr_rows = []

for method in labeled_df["chunking_method"].unique():

    method_df = labeled_df[
        labeled_df["chunking_method"] == method
    ]

    rr_scores = []

    for question_id, group in method_df.groupby("question_id"):
        rr_scores.append(
            reciprocal_rank_at_k(group, k=TOP_K)
        )

    mrr_rows.append({
        "chunking_method": method,
        f"MRR@{TOP_K}": sum(rr_scores) / len(rr_scores)
    })


### Code 133 — Prepare mrr_df


In [ ]:
mrr_df = pd.DataFrame(mrr_rows)

display(
    mrr_df.round(3)
)

,chunking_method,MRR@10
0,recursive,0.975
1,slm_assisted,0.967


## Common-Pool nDCG@K

\[
DCG@K=\sum_{i=1}^{K}
\frac{2^{rel_i}-1}{\log_2(i+1)}
\]

The corrected notebook constructs a **common judged candidate pool** from the union of both methods' top-10 candidates for each question. `Pooled IDCG@K` is calculated from that common pool.

\[
Pooled\ nDCG@K = \frac{DCG@K}{Pooled\ IDCG@K}
\]

This is still a **top-10 pooled** evaluation, not exhaustive full-corpus nDCG with complete qrels.


### Code 134 — Common-Pool nDCG@K


In [ ]:
import numpy as np

### Code 135 — Define dcg_at_k


In [ ]:
def dcg_at_k(relevance_scores, k):

    relevance_scores = np.array(
        relevance_scores[:k],
        dtype=float
    )

    if len(relevance_scores) == 0:
        return 0.0

    discounts = np.log2(
        np.arange(2, len(relevance_scores) + 2)
    )

    gains = (
        (2 ** relevance_scores) - 1
    )

    return np.sum(
        gains / discounts
    )

### Code 136 — Define pooled_qrels


In [ ]:
pooled_qrels = {}

for question_id, qdf in labeled_df.groupby("question_id"):

    pooled = (
        qdf[["chunk_id", "relevance_label"]]
        .groupby("chunk_id", as_index=False)["relevance_label"]
        .max()
    )

    pooled_qrels[question_id] = pooled


def pooled_ndcg_at_k(method_group, question_id, k):

    method_group = method_group.sort_values("rank")

    actual_relevance = method_group["relevance_label"].tolist()
    dcg = dcg_at_k(actual_relevance, k)

    pooled_relevance = pooled_qrels[question_id]["relevance_label"].tolist()
    ideal_relevance = sorted(pooled_relevance, reverse=True)
    idcg = dcg_at_k(ideal_relevance, k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


### Code 137 — Define ndcg_rows


In [ ]:
ndcg_rows = []

for method in labeled_df["chunking_method"].unique():

    method_df = labeled_df[
        labeled_df["chunking_method"] == method
    ]

    for k in K_VALUES:

        question_scores = []

        for question_id, group in method_df.groupby("question_id"):
            question_scores.append(
                pooled_ndcg_at_k(group, question_id, k)
            )

        ndcg_rows.append({
            "chunking_method": method,
            "k": k,
            "pooled_ndcg": sum(question_scores) / len(question_scores)
        })


### Code 138 — Prepare ndcg_df


In [ ]:
ndcg_df = pd.DataFrame(ndcg_rows)
display(ndcg_df.round(3))


,chunking_method,k,pooled_ndcg
0,recursive,1,0.783
1,recursive,3,0.697
2,recursive,5,0.720
3,recursive,10,0.748
4,slm_assisted,1,0.783
5,slm_assisted,3,0.727
6,slm_assisted,5,0.676
7,slm_assisted,10,0.659


## Cross-Method Pooled Recall@K

The original notebook used each method's own top-10 relevant count as the denominator, which forced Recall@10 to 1.0. This corrected version uses the **combined judged pool from both methods**:

\[
PooledRecall@K =
\frac{\#\text{ relevant unique chunks retrieved by the method in top }K}
{\#\text{ relevant unique chunks in the combined judged pool}}
\]

This is **not exhaustive full-corpus Recall@K**.


### Code 139 — Cross-Method Pooled Recall@K


In [ ]:
def pooled_recall_at_k(method_group, question_id, k):

    method_group = method_group.sort_values("rank").head(k)
    pooled = pooled_qrels[question_id]

    pooled_relevant_ids = set(
        pooled.loc[
            pooled["relevance_label"] >= 1,
            "chunk_id"
        ]
    )

    if len(pooled_relevant_ids) == 0:
        return 0.0

    retrieved_relevant_ids = set(
        method_group.loc[
            method_group["is_relevant"] == 1,
            "chunk_id"
        ]
    )

    return (
        len(retrieved_relevant_ids & pooled_relevant_ids)
        / len(pooled_relevant_ids)
    )


### Code 140 — Define recall_rows


In [ ]:
recall_rows = []

for method in labeled_df["chunking_method"].unique():

    method_df = labeled_df[
        labeled_df["chunking_method"] == method
    ]

    for k in K_VALUES:

        question_scores = []

        for question_id, group in method_df.groupby("question_id"):
            question_scores.append(
                pooled_recall_at_k(group, question_id, k)
            )

        recall_rows.append({
            "chunking_method": method,
            "k": k,
            "pooled_recall": sum(question_scores) / len(question_scores)
        })


### Code 141 — Prepare recall_df


In [ ]:
recall_df = pd.DataFrame(
    recall_rows
)

display(
    recall_df.round(3)
)

,chunking_method,k,pooled_recall
0,recursive,1,0.056
1,recursive,3,0.162
2,recursive,5,0.271
3,recursive,10,0.528
4,slm_assisted,1,0.057
5,slm_assisted,3,0.162
6,slm_assisted,5,0.246
7,slm_assisted,10,0.472


Step 1 — Pivot Precision results

### Code 142 — Step 1 — Pivot Precision results


In [ ]:
precision_pivot = precision_df.pivot(
    index="chunking_method",
    columns="k",
    values="precision"
)

precision_pivot.columns = [
    f"Precision@{k}"
    for k in precision_pivot.columns
]

display(
    precision_pivot.round(3)
)

,Precision@1,Precision@3,Precision@5,Precision@10
chunking_method,,,,
recursive,0.95,0.917,0.92,0.90
slm_assisted,0.95,0.917,0.84,0.81


Step 2 — Pivot common-pool nDCG results.

### Code 143 — Step 2 — Pivot common-pool nDCG results.


In [ ]:
ndcg_pivot = ndcg_df.pivot(
    index="chunking_method",
    columns="k",
    values="pooled_ndcg"
)

ndcg_pivot.columns = [
    f"PoolednDCG@{k}"
    for k in ndcg_pivot.columns
]

display(ndcg_pivot.round(3))


,PoolednDCG@1,PoolednDCG@3,PoolednDCG@5,PoolednDCG@10
chunking_method,,,,
recursive,0.783,0.697,0.720,0.748
slm_assisted,0.783,0.727,0.676,0.659


Step 3 — Pivot cross-method pooled Recall results.

### Code 144 — Step 3 — Pivot cross-method pooled Recall results.


In [ ]:
recall_pivot = recall_df.pivot(
    index="chunking_method",
    columns="k",
    values="pooled_recall"
)

recall_pivot.columns = [
    f"PooledRecall@{k}"
    for k in recall_pivot.columns
]

display(
    recall_pivot.round(3)
)

,PooledRecall@1,PooledRecall@3,PooledRecall@5,PooledRecall@10
chunking_method,,,,
recursive,0.056,0.162,0.271,0.528
slm_assisted,0.057,0.162,0.246,0.472


Step 4 — Prepare MRR@10.

### Code 145 — Step 4 — Prepare MRR@10.


In [ ]:
mrr_summary = (
    mrr_df
    .set_index("chunking_method")
)

display(
    mrr_summary.round(3)
)

,MRR@10
chunking_method,
recursive,0.975
slm_assisted,0.967


Step 5 — Combine everything

### Code 146 — Step 5 — Combine everything


In [ ]:
final_retrieval_comparison = pd.concat(
    [
        precision_pivot,
        mrr_summary,
        ndcg_pivot,
        recall_pivot
    ],
    axis=1
)

display(
    final_retrieval_comparison.round(3)
)

,Precision@1,Precision@3,Precision@5,Precision@10,MRR@10,PoolednDCG@1,PoolednDCG@3,PoolednDCG@5,PoolednDCG@10,PooledRecall@1,PooledRecall@3,PooledRecall@5,PooledRecall@10
chunking_method,,,,,,,,,,,,,
recursive,0.95,0.917,0.92,0.90,0.975,0.783,0.697,0.720,0.748,0.056,0.162,0.271,0.528
slm_assisted,0.95,0.917,0.84,0.81,0.967,0.783,0.727,0.676,0.659,0.057,0.162,0.246,0.472


Step 6 — Add the study-specific Boundary Quality Score as supplementary evidence.

### Code 147 — Step 6 — Add the study-specific Boundary Quality Score as supplementary evidence.


In [ ]:
bqs_by_method = {
    "recursive": float(recursive_score),
    "slm_assisted": float(slm_score)
}

final_retrieval_comparison["BoundaryQuality"] = [
    bqs_by_method.get(method, np.nan)
    for method in final_retrieval_comparison.index
]


### Code 148 — Prepare boundary_relative_improvement


In [ ]:
boundary_relative_improvement = relative_improvement

print(
    "SLM BQS relative improvement:",
    round(boundary_relative_improvement, 2),
    "%"
)


SLM BQS relative improvement: 27.12 %


### Code 149 — Display Results


In [ ]:
display(
    final_retrieval_comparison.round(3)
)

,Precision@1,Precision@3,Precision@5,Precision@10,MRR@10,PoolednDCG@1,PoolednDCG@3,PoolednDCG@5,PoolednDCG@10,PooledRecall@1,PooledRecall@3,PooledRecall@5,PooledRecall@10,BoundaryQuality
chunking_method,,,,,,,,,,,,,,
recursive,0.95,0.917,0.92,0.90,0.975,0.783,0.697,0.720,0.748,0.056,0.162,0.271,0.528,0.708
slm_assisted,0.95,0.917,0.84,0.81,0.967,0.783,0.727,0.676,0.659,0.057,0.162,0.246,0.472,0.900


Step 7 — Save the comparison table

### Code 150 — Step 7 — Save the comparison table


In [ ]:
final_comparison_path = os.path.join(
    EVALUATION_FOLDER,
    "final_recursive_vs_slm_comparison.csv"
)

final_retrieval_comparison.to_csv(
    final_comparison_path
)

print(
    "Saved:",
    final_comparison_path
)

Saved: /content/drive/MyDrive/pcos_rag/evaluation/final_recursive_vs_slm_comparison.csv


## Record the Final Chunking Decision

No arbitrary weighted composite score is invented. The decision is based on the pattern across study-specific BQS and multiple early-rank retrieval metrics. Common-pool nDCG and pooled Recall are limited to the judged top-10 candidate pool.


### Code 151 — Record the Final Chunking Decision


In [ ]:
FINAL_CHUNKING_METHOD = "slm_assisted"

primary_metric_columns = [
    col
    for col in [
        "Precision@1",
        "Precision@3",
        f"MRR@{TOP_K}",
        "PoolednDCG@3",
        "PoolednDCG@5",
        "PooledRecall@3",
        "PooledRecall@5",
        "BoundaryQuality"
    ]
    if col in final_retrieval_comparison.columns
]

metric_winners = {
    metric: final_retrieval_comparison[metric].idxmax()
    for metric in primary_metric_columns
}

print("Primary metric winners:")
for metric, winner in metric_winners.items():
    print(f"{metric}: {winner}")

selection_reason = {
    "selected_method": FINAL_CHUNKING_METHOD,
    "reason": (
        "SLM-assisted chunking was selected after considering the "
        "study-specific BQS together with multiple early-rank retrieval "
        "metrics. No arbitrary weighted composite score was used. "
        "Pooled nDCG and Recall are limited to the union of the judged "
        "top-10 candidates from both chunking methods."
    )
}

selection_reason


Primary metric winners:
Precision@1: recursive
Precision@3: recursive
MRR@10: recursive
PoolednDCG@3: slm_assisted
PoolednDCG@5: recursive
PooledRecall@3: slm_assisted
PooledRecall@5: recursive
BoundaryQuality: slm_assisted


{'selected_method': 'slm_assisted',
 'reason': 'SLM-assisted chunking was selected after considering the study-specific BQS together with multiple early-rank retrieval metrics. No arbitrary weighted composite score was used. Pooled nDCG and Recall are limited to the union of the judged top-10 candidates from both chunking methods.'}

Step 2 — Save the decision

### Code 152 — Step 2 — Save the decision


In [ ]:
selection_df = pd.DataFrame(
    [selection_reason]
)

selection_path = os.path.join(
    EVALUATION_FOLDER,
    "final_chunking_method_selection.csv"
)

selection_df.to_csv(
    selection_path,
    index=False
)

print(
    "Selected method:",
    FINAL_CHUNKING_METHOD
)

print(
    "Saved:",
    selection_path
)

Selected method: slm_assisted
Saved: /content/drive/MyDrive/pcos_rag/evaluation/final_chunking_method_selection.csv


Step 3 — Make SLM the final corpus

### Code 153 — Step 3 — Make SLM the final corpus


In [ ]:
final_chunks = slm_retrieval_chunks

print(
    "Final chunking method:",
    FINAL_CHUNKING_METHOD
)

print(
    "Final chunks:",
    len(final_chunks)
)

Final chunking method: slm_assisted
Final chunks: 932


Step 4 — Convert final chunks into a DataFrame

### Code 154 — Step 4 — Convert final chunks into a DataFrame


In [ ]:
final_chunk_rows = []

for chunk in final_chunks:

    final_chunk_rows.append({
        "chunk_id": chunk.metadata.get("chunk_id"),
        "document_name": chunk.metadata.get("document_name"),
        "pdf_page_number": chunk.metadata.get("pdf_page_number"),
        "organization": chunk.metadata.get("organization"),
        "source_type": chunk.metadata.get("source_type"),
        "extraction_method": chunk.metadata.get("extraction_method"),
        "chunking_method": chunk.metadata.get("chunking_method"),
        "page_chunk_index": chunk.metadata.get("page_chunk_index"),
        "word_count": len(chunk.page_content.split()),
        "character_count": len(chunk.page_content),
        "retrieval_eligible": True,
        "eligibility_reason": "eligible",
        "text": chunk.page_content,
    })

final_chunks_df = pd.DataFrame(
    final_chunk_rows
)

# Direct handoff to Qdrant — no second cleaning/chunking rebuild.
final_retrieval_df = (
    final_chunks_df
    .copy()
    .reset_index(drop=True)
)

print(
    "Final knowledge-base chunks:",
    len(final_chunks_df)
)

print(
    "final_retrieval_df ready for Qdrant:",
    len(final_retrieval_df)
)

display(
    final_chunks_df.head()
)


Final knowledge-base chunks: 932
final_retrieval_df ready for Qdrant: 932


,chunk_id,document_name,pdf_page_number,organization,source_type,extraction_method,chunking_method,page_chunk_index,word_count,character_count,retrieval_eligible,eligibility_reason,text
0,SLM_CHUNK_00000,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,docling_layout_aware,slm_assisted,0,146,938,True,eligible,Eunice Kennedy Shriver National Institute of C...
1,SLM_CHUNK_00001,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,docling_layout_aware,slm_assisted,1,80,562,True,eligible,How does NICHD support PCOS research? NICHD ha...
2,SLM_CHUNK_00002,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,docling_layout_aware,slm_assisted,2,85,557,True,eligible,Success Snapshots Clarifying the Role of Genet...
3,SLM_CHUNK_00003,NICHDWHPCOSweb.pdf,1,NICHD,patient_health_information,docling_layout_aware,slm_assisted,3,82,560,True,eligible,Defining Subtypes of PCOS Because symptoms and...
4,SLM_CHUNK_00004,NICHDWHPCOSweb.pdf,2,NICHD,patient_health_information,docling_layout_aware,slm_assisted,0,88,647,True,eligible,Selected NICHD-Funded Polycystic Ovary Syndrom...


Step 5 — Save the final selected corpus

### Code 155 — Step 5 — Save the final selected corpus


In [ ]:
final_chunks_path = os.path.join(
    CHUNKS_FOLDER,
    "final_selected_slm_chunks.csv"
)

final_chunks_df.to_csv(
    final_chunks_path,
    index=False
)

print(
    "Final corpus saved:",
    final_chunks_path
)

Final corpus saved: /content/drive/MyDrive/pcos_rag/chunks/final_selected_slm_chunks.csv


### Code 156 — Prepare final_jsonl_path


In [ ]:
final_jsonl_path = os.path.join(
    CHUNKS_FOLDER,
    "final_selected_slm_chunks.jsonl"
)

final_chunks_df.to_json(
    final_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "JSONL saved:",
    final_jsonl_path
)

JSONL saved: /content/drive/MyDrive/pcos_rag/chunks/final_selected_slm_chunks.jsonl


### Final Qdrant Build — use the already-cleaned, already-chunked final corpus

This cell does **not** clean or chunk again. It only embeds `final_retrieval_df`, saves artifacts, and syncs the same corpus to Qdrant Cloud.


In [ ]:
# ============================================================
# FINAL CORPUS -> MINILM EMBEDDINGS -> QDRANT CLOUD
# MiniLM is used ONLY for dense embeddings.
# ============================================================

from pathlib import Path
import json
import os
import uuid

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams


# ============================================================
# 1. FINAL CORPUS
# ============================================================

if "final_retrieval_df" not in globals():
    raise RuntimeError(
        "final_retrieval_df is missing. "
        "Run the notebook through the final SLM corpus selection cell first."
    )

retrieval_corpus_df = (
    final_retrieval_df
    .copy()
    .reset_index(drop=True)
)

if len(retrieval_corpus_df) == 0:
    raise RuntimeError("No retrieval-eligible chunks are available.")

assert retrieval_corpus_df["chunk_id"].astype(str).is_unique
assert retrieval_corpus_df["text"].fillna("").astype(str).str.strip().ne("").all()

if "pdf_page_number" in retrieval_corpus_df.columns:
    retrieval_corpus_df["page"] = retrieval_corpus_df["pdf_page_number"]


def qdrant_point_id_from_chunk_id(chunk_id):
    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"pcos-layout-rag:{chunk_id}",
        )
    )


retrieval_corpus_df["qdrant_point_id"] = (
    retrieval_corpus_df["chunk_id"]
    .astype(str)
    .map(qdrant_point_id_from_chunk_id)
)

assert retrieval_corpus_df["qdrant_point_id"].is_unique


# ============================================================
# 2. MINILM EMBEDDINGS ONLY
# ============================================================

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

retrieval_embeddings = (
    embedding_model.encode(
        retrieval_corpus_df["text"].astype(str).tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    .astype("float32")
)

embedding_dimension = int(retrieval_embeddings.shape[1])
assert embedding_dimension == 384
assert np.allclose(
    np.linalg.norm(retrieval_embeddings, axis=1),
    1.0,
    atol=1e-3,
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding shape:", retrieval_embeddings.shape)


# ============================================================
# 3. SAVE LOCAL/DRIVE ARTIFACTS
# ============================================================

VECTOR_FOLDER = Path(
    "/content/drive/MyDrive/pcos_rag/vector_store"
)

QDRANT_ARTIFACT_DIR = (
    VECTOR_FOLDER / "qdrant_layout_cleaned"
)
QDRANT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_CSV_PATH = QDRANT_ARTIFACT_DIR / "retrieval_corpus.csv"
CORPUS_JSONL_PATH = QDRANT_ARTIFACT_DIR / "retrieval_corpus.jsonl"
EMBEDDINGS_PATH = QDRANT_ARTIFACT_DIR / "retrieval_embeddings.npy"
CONFIG_PATH = QDRANT_ARTIFACT_DIR / "qdrant_retrieval_config.json"

retrieval_corpus_df.to_csv(CORPUS_CSV_PATH, index=False)
retrieval_corpus_df.to_json(
    CORPUS_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)
np.save(EMBEDDINGS_PATH, retrieval_embeddings)


# ============================================================
# 4. QDRANT CLOUD SETTINGS
# ============================================================

# Cluster endpoint is not a secret. Change it here only if the cluster changes.
QDRANT_URL = (
    "https://74b795c6-af27-4520-b2fe-07363d53b8ea."
    "us-east-1-1.aws.cloud.qdrant.io"
)

# Keep the API key OUT of notebook source.
# Recommended in Colab: create a Secret named QDRANT_API_KEY.
# Local alternative: export QDRANT_API_KEY="..." before running Python.
QDRANT_COLLECTION = "pcos_layout_cleaned_v1"
QDRANT_SYNC_ENABLED = True
QDRANT_BATCH_SIZE = 128


def get_qdrant_api_key():
    key = os.getenv("QDRANT_API_KEY")

    if key:
        return key.strip()

    try:
        from google.colab import userdata
        key = userdata.get("QDRANT_API_KEY")
        if key:
            return str(key).strip()
    except Exception:
        pass

    return None


def connect_qdrant_cloud():
    api_key = get_qdrant_api_key()

    if not api_key:
        raise RuntimeError(
            "QDRANT_API_KEY is not set. In Colab, add it under "
            "Secrets with the name QDRANT_API_KEY."
        )

    client = QdrantClient(
        url=QDRANT_URL,
        api_key=api_key,
        timeout=60,
    )

    # Lightweight authenticated connectivity check.
    client.get_collections()
    return client


def ensure_qdrant_collection(client):
    if not client.collection_exists(
        collection_name=QDRANT_COLLECTION
    ):
        client.create_collection(
            collection_name=QDRANT_COLLECTION,
            vectors_config=VectorParams(
                size=embedding_dimension,
                distance=Distance.COSINE,
            ),
        )
        print("Created Qdrant collection:", QDRANT_COLLECTION)
    else:
        print("Using existing Qdrant collection:", QDRANT_COLLECTION)


def sync_corpus_to_qdrant(client):
    ensure_qdrant_collection(client)

    for start in range(0, len(retrieval_corpus_df), QDRANT_BATCH_SIZE):
        stop = min(start + QDRANT_BATCH_SIZE, len(retrieval_corpus_df))
        points = []

        for row_idx in range(start, stop):
            row = retrieval_corpus_df.iloc[row_idx]

            payload = {
                "chunk_id": str(row["chunk_id"]),
                "document_name": row.get("document_name"),
                "page": (
                    int(row["page"])
                    if pd.notna(row.get("page"))
                    else None
                ),
                "organization": row.get("organization"),
                "source_type": row.get("source_type"),
                "extraction_method": row.get("extraction_method"),
                "chunking_method": row.get("chunking_method"),
                "text": str(row["text"]),
            }

            # Remove NaN values from payload.
            payload = {
                key: value
                for key, value in payload.items()
                if value is not None
                and not (
                    isinstance(value, float)
                    and np.isnan(value)
                )
            }

            points.append(
                PointStruct(
                    id=str(row["qdrant_point_id"]),
                    vector=retrieval_embeddings[row_idx].tolist(),
                    payload=payload,
                )
            )

        client.upsert(
            collection_name=QDRANT_COLLECTION,
            points=points,
            wait=True,
        )

        print(f"Upserted {stop}/{len(retrieval_corpus_df)} points")

    count_result = client.count(
        collection_name=QDRANT_COLLECTION,
        exact=True,
    )
    print("Qdrant collection count:", count_result.count)


QDRANT_CONFIG = {
    "vector_store": "qdrant_cloud",
    "cluster_url": QDRANT_URL,
    "collection_name": QDRANT_COLLECTION,
    "corpus_policy": "layout_cleaned_structurally_eligible_chunks",
    "document_parser": "docling_layout_aware_with_pypdf_fallback",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": embedding_dimension,
    "normalize_embeddings": True,
    "distance": "cosine",
    "lexical_retrieval": "BM25Okapi",
    "fusion": "Reciprocal Rank Fusion",
    "cross_encoder_reranking": True,
    "row_count": len(retrieval_corpus_df),
    "point_id_strategy": "UUID5(stable chunk id)",
    "api_key_storage": "Colab Secret or environment variable",
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(QDRANT_CONFIG, f, indent=2)


# ============================================================
# 5. CONNECT + SYNC
# ============================================================

qdrant_client = None

if QDRANT_SYNC_ENABLED:
    try:
        qdrant_client = connect_qdrant_cloud()
        print("✅ Connected to Qdrant Cloud")
        sync_corpus_to_qdrant(qdrant_client)
        print("✅ Qdrant Cloud sync complete")
    except RuntimeError as exc:
        print("Qdrant sync skipped:", exc)
        print("Set QDRANT_API_KEY and rerun this cell to connect/sync.")


# ============================================================
# 6. ROUND-TRIP LOCAL VALIDATION
# ============================================================

loaded_corpus = pd.read_csv(CORPUS_CSV_PATH)
loaded_embeddings = np.load(EMBEDDINGS_PATH)

assert len(loaded_corpus) == len(retrieval_corpus_df)
assert loaded_embeddings.shape == retrieval_embeddings.shape
assert loaded_corpus["chunk_id"].astype(str).is_unique
assert loaded_corpus["qdrant_point_id"].astype(str).is_unique

print("\n" + "=" * 80)
print("LAYOUT-AWARE QDRANT ARTIFACT BUILD COMPLETE")
print("=" * 80)
print("Final chunks:", len(retrieval_corpus_df))
print("Embedding dimension:", embedding_dimension)
print("Qdrant collection:", QDRANT_COLLECTION)
print("Saved:", QDRANT_ARTIFACT_DIR)


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (932, 384)
✅ Connected to Qdrant Cloud
Using existing Qdrant collection: pcos_layout_cleaned_v1
Upserted 128/932 points
Upserted 256/932 points
Upserted 384/932 points
Upserted 512/932 points
Upserted 640/932 points
Upserted 768/932 points
Upserted 896/932 points
Upserted 932/932 points
Qdrant collection count: 932
✅ Qdrant Cloud sync complete

LAYOUT-AWARE QDRANT ARTIFACT BUILD COMPLETE
Final chunks: 932
Embedding dimension: 384
Qdrant collection: pcos_layout_cleaned_v1
Saved: /content/drive/MyDrive/pcos_rag/vector_store/qdrant_layout_cleaned


### Handoff to the final XAI + Meta-Adaptive RAG notebook

Use the Qdrant Cloud collection `pcos_layout_cleaned_v1` as the dense store and rebuild BM25 over the **same corpus rows**. Keep RRF fusion and Cross-Encoder reranking. The downstream notebook should consume this corpus directly rather than applying an additional semantic pre-filter.


## Manual/New PDF Input Pipeline — Same Global Cleaner

This section processes one new PDF with **the exact same parser and cleaning path as the full corpus**. It is intended for testing a newly added verified source before or alongside a full corpus rebuild.


### Code 332 — Define Manual/New PDF Processing Helpers

In [ ]:
# ============================================================
# MANUAL / NEW PDF INPUT HELPERS
# Uses the SAME global layout-aware cleaning path as batch ingestion.
# ============================================================

from pathlib import Path
import uuid
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer


MANUAL_PIPELINE_REQUIRED_OBJECTS = [
    "load_and_clean_pdf",
    "build_slm_chunks_for_page",
    "assess_retrieval_eligibility",
    "stable_chunk_id",
]

manual_missing = [
    name
    for name in MANUAL_PIPELINE_REQUIRED_OBJECTS
    if name not in globals()
]

if manual_missing:
    raise RuntimeError(
        "Run the global cleaning/chunking setup first. "
        f"Missing objects: {manual_missing}"
    )

MANUAL_EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer(
        MANUAL_EMBEDDING_MODEL_NAME
    )


def process_manual_pdf(
    pdf_path,
    organization="manual_upload",
    source_type="manual_upload",
):
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(pdf_path)

    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("Manual input must be a PDF.")

    filename = pdf_path.name

    document_result = load_and_clean_pdf(
        pdf_path=pdf_path,
        organization=organization,
        source_type=source_type,
        manual_input=True,
    )

    raw_manual_pages = document_result["raw_pages"]
    cleaned_manual_pages = document_result["cleaned_pages"]
    excluded_pages = document_result["excluded_pages"]

    manual_chunks = []

    for page in cleaned_manual_pages:
        page_number = page.metadata["pdf_page_number"]
        page_chunks = build_slm_chunks_for_page(page.page_content)

        for local_index, chunk_text in enumerate(page_chunks):
            chunk_id = stable_chunk_id(
                document_name=filename,
                page_number=page_number,
                page_chunk_index=local_index,
                chunk_text=chunk_text,
            )

            metadata = page.metadata.copy()
            metadata.update({
                "chunking_method": "slm_assisted",
                "chunk_id": chunk_id,
                "page_chunk_index": local_index,
                "character_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
                "manual_input": True,
            })

            manual_chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata=metadata,
                )
            )

    eligible_rows = []
    excluded_chunks = []

    for chunk in manual_chunks:
        eligible, reason = assess_retrieval_eligibility(
            chunk.page_content
        )

        if eligible:
            eligible_rows.append({
                **chunk.metadata,
                "retrieval_eligible": True,
                "eligibility_reason": reason,
                "text": chunk.page_content,
            })
        else:
            excluded_chunks.append({
                "chunk_id": chunk.metadata.get("chunk_id"),
                "document_name": filename,
                "pdf_page_number": chunk.metadata.get("pdf_page_number"),
                "reason": reason,
                "text": chunk.page_content,
            })

    eligible_df = pd.DataFrame(eligible_rows)

    if len(eligible_df) > 0:
        manual_embeddings = (
            embedding_model.encode(
                eligible_df["text"].astype(str).tolist(),
                batch_size=64,
                show_progress_bar=True,
                normalize_embeddings=True,
                convert_to_numpy=True,
            )
            .astype("float32")
        )
        assert manual_embeddings.shape == (len(eligible_df), 384)
    else:
        manual_embeddings = np.empty((0, 384), dtype="float32")

    return {
        "document_name": filename,
        "raw_page_count": len(raw_manual_pages),
        "clean_page_count": len(cleaned_manual_pages),
        "excluded_pages": pd.DataFrame(excluded_pages),
        "chunk_count": len(manual_chunks),
        "eligible_chunk_count": len(eligible_df),
        "excluded_chunks": pd.DataFrame(excluded_chunks),
        "eligible_df": eligible_df,
        "embeddings": manual_embeddings,
        "extraction_report": document_result["extraction_report"],
        "repeated_edge_report": document_result["repeated_edge_report"],
    }


print("✅ Manual/new PDF helpers use the same global cleaner.")


✅ Manual/new PDF helpers use the same global cleaner.


### Code 333 — Manually Upload One PDF and Run the Shared Pipeline

In [ ]:
# ============================================================
# MANUAL / NEW PDF INPUT
# Upload one PDF and run the same global ingestion pipeline.
# ============================================================

from google.colab import files
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

print("Upload ONE verified PDF document for pipeline testing.")
uploaded = files.upload()

pdf_names = [
    name for name in uploaded.keys()
    if name.lower().endswith(".pdf")
]

if len(pdf_names) == 0:
    raise RuntimeError("No PDF file was uploaded.")
if len(pdf_names) > 1:
    raise RuntimeError("Please upload only one PDF at a time.")

manual_pdf_name = pdf_names[0]
manual_pdf_path = Path(manual_pdf_name)

manual_organization = input(
    "Organization/source name (press Enter for 'manual_upload'): "
).strip() or "manual_upload"

manual_source_type = input(
    "Source type (e.g. guideline, official_web_resource; "
    "press Enter for 'manual_upload'): "
).strip() or "manual_upload"

manual_pipeline_result = process_manual_pdf(
    pdf_path=manual_pdf_path,
    organization=manual_organization,
    source_type=manual_source_type,
)

manual_stem = Path(
    manual_pipeline_result["document_name"]
).stem
safe_stem = re.sub(
    r"[^A-Za-z0-9._-]+",
    "_",
    manual_stem,
).strip("_")

manual_output_dir = (
    Path(
        "/content/drive/MyDrive/pcos_rag/"
        "manual_document_ingestion_layout_cleaned"
    ) / safe_stem
)
manual_output_dir.mkdir(parents=True, exist_ok=True)

eligible_path = manual_output_dir / "manual_retrieval_eligible_chunks.csv"
eligible_jsonl_path = manual_output_dir / "manual_retrieval_eligible_chunks.jsonl"
embeddings_path = manual_output_dir / "manual_retrieval_embeddings.npy"
excluded_pages_path = manual_output_dir / "manual_excluded_pages.csv"
excluded_chunks_path = manual_output_dir / "manual_excluded_chunks.csv"
manifest_path = manual_output_dir / "manual_pipeline_manifest.json"

manual_pipeline_result["eligible_df"].to_csv(
    eligible_path,
    index=False,
)
manual_pipeline_result["eligible_df"].to_json(
    eligible_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False,
)
np.save(
    embeddings_path,
    manual_pipeline_result["embeddings"],
)
manual_pipeline_result["excluded_pages"].to_csv(
    excluded_pages_path,
    index=False,
)
manual_pipeline_result["excluded_chunks"].to_csv(
    excluded_chunks_path,
    index=False,
)

manifest = {
    "document_name": manual_pipeline_result["document_name"],
    "organization": manual_organization,
    "source_type": manual_source_type,
    "document_parser": manual_pipeline_result[
        "extraction_report"
    ].get("parser"),
    "parser_fallback_used": manual_pipeline_result[
        "extraction_report"
    ].get("fallback_used"),
    "repeated_edge_lines_removed": manual_pipeline_result[
        "repeated_edge_report"
    ].get("removed_line_count", 0),
    "corpus_policy": "layout_cleaned_structurally_eligible_chunks",
    "raw_page_count": manual_pipeline_result["raw_page_count"],
    "clean_page_count": manual_pipeline_result["clean_page_count"],
    "chunk_count": manual_pipeline_result["chunk_count"],
    "eligible_chunk_count": manual_pipeline_result["eligible_chunk_count"],
    "embedding_model": MANUAL_EMBEDDING_MODEL_NAME,
    "embedding_dimension": 384,
    "normalize_embeddings": True,
    "qdrant_ready": True,
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 80)
print("MANUAL / NEW PDF PIPELINE COMPLETE")
print("=" * 80)
print("Document:", manual_pipeline_result["document_name"])
print("Parser:", manual_pipeline_result["extraction_report"].get("parser"))
print("Raw pages:", manual_pipeline_result["raw_page_count"])
print("Clean pages:", manual_pipeline_result["clean_page_count"])
print("SLM chunks:", manual_pipeline_result["chunk_count"])
print("Final retrieval-eligible chunks:", manual_pipeline_result["eligible_chunk_count"])
print("Embedding shape:", manual_pipeline_result["embeddings"].shape)
print("Saved review artifacts to:", manual_output_dir)

if len(manual_pipeline_result["excluded_pages"]) > 0:
    print("\nExcluded pages:")
    display(manual_pipeline_result["excluded_pages"])

if len(manual_pipeline_result["excluded_chunks"]) > 0:
    print("\nExcluded chunks:")
    display(manual_pipeline_result["excluded_chunks"])

if len(manual_pipeline_result["eligible_df"]) > 0:
    display(
        manual_pipeline_result["eligible_df"][[
            col for col in [
                "chunk_id",
                "pdf_page_number",
                "extraction_method",
                "retrieval_eligible",
                "eligibility_reason",
                "text",
            ]
            if col in manual_pipeline_result["eligible_df"].columns
        ]].head(10)
    )

# ============================================================
# 6. OPTIONAL QDRANT SYNC FOR MANUAL UPLOADS
# ============================================================

# Check if Qdrant client functions are available (from the main pipeline)
if 'connect_qdrant_cloud' in globals() and 'ensure_qdrant_collection' in globals() and 'qdrant_point_id_from_chunk_id' in globals():
    try:
        # Establish connection to Qdrant if not already connected globally
        if 'qdrant_client' not in globals() or qdrant_client is None:
            qdrant_client = connect_qdrant_cloud()
            print("✅ Reconnected to Qdrant Cloud for manual upload.")

        # Define a collection name for manual uploads, distinct from the main corpus
        MANUAL_QDRANT_COLLECTION = "pcos_manual_uploads_v1"

        # Ensure the collection exists
        # We need the embedding_dimension from the main pipeline's setup
        if 'embedding_dimension' not in globals():
            embedding_dimension = manual_pipeline_result["embeddings"].shape[1] # Assume 384 if not defined

        # Override QDRANT_COLLECTION for this call to create the manual collection
        original_qdrant_collection = globals().get('QDRANT_COLLECTION', None)
        globals()['QDRANT_COLLECTION'] = MANUAL_QDRANT_COLLECTION
        ensure_qdrant_collection(qdrant_client)
        if original_qdrant_collection is not None:
            globals()['QDRANT_COLLECTION'] = original_qdrant_collection # Restore original


        manual_eligible_df = manual_pipeline_result["eligible_df"]
        manual_embeddings = manual_pipeline_result["embeddings"]

        if len(manual_eligible_df) > 0:
            points = []
            for idx, row in manual_eligible_df.iterrows():
                payload = {
                    "chunk_id": str(row["chunk_id"]),
                    "document_name": row.get("document_name"),
                    "page": (
                        int(row["pdf_page_number"])
                        if pd.notna(row.get("pdf_page_number"))
                        else None
                    ),
                    "organization": row.get("organization"),
                    "source_type": row.get("source_type"),
                    "extraction_method": row.get("extraction_method"),
                    "chunking_method": row.get("chunking_method"),
                    "text": str(row["text"]),
                    "manual_input": True, # Flag for manual input
                }

                # Remove NaN values from payload.
                payload = {
                    key: value
                    for key, value in payload.items()
                    if value is not None
                    and not (
                        isinstance(value, float)
                        and np.isnan(value)
                    )
                }

                points.append(
                    PointStruct(
                        id=qdrant_point_id_from_chunk_id(str(row["chunk_id"])),
                        vector=manual_embeddings[idx].tolist(),
                        payload=payload,
                    )
                )

            # Upsert in batches (reusing QDRANT_BATCH_SIZE if defined, otherwise a default)
            QDRANT_BATCH_SIZE_MANUAL = globals().get('QDRANT_BATCH_SIZE', 128)
            for start_batch in range(0, len(points), QDRANT_BATCH_SIZE_MANUAL):
                end_batch = min(start_batch + QDRANT_BATCH_SIZE_MANUAL, len(points))
                batch_points = points[start_batch:end_batch]
                qdrant_client.upsert(
                    collection_name=MANUAL_QDRANT_COLLECTION,
                    points=batch_points,
                    wait=True,
                )
                print(f"Upserted {end_batch}/{len(points)} manual points to {MANUAL_QDRANT_COLLECTION}")

            count_result = qdrant_client.count(
                collection_name=MANUAL_QDRANT_COLLECTION,
                exact=True,
            )
            print(f"Qdrant manual collection count: {count_result.count}")
            print(f"✅ Manual document '{manual_pdf_name}' synced to Qdrant collection '{MANUAL_QDRANT_COLLECTION}'.")
        else:
            print(f"No eligible chunks to sync for manual document '{manual_pdf_name}'.")

    except RuntimeError as exc:
        print("Qdrant sync for manual upload skipped:", exc)
        print("Ensure QDRANT_API_KEY is set in Colab Secrets or environment variables.")
    except Exception as exc:
        print(f"An unexpected error occurred during Qdrant sync for manual upload: {exc}")
else:
    print("Qdrant client functions (connect_qdrant_cloud, ensure_qdrant_collection, qdrant_point_id_from_chunk_id) not found. Run the main Qdrant setup cell first.")

Upload ONE verified PDF document for pipeline testing.


RuntimeError: No PDF file was uploaded.

### Execution order for the final ingestion path

1. Mount Drive, define folders, install dependencies, and load source metadata.
2. Run the **global layout-aware extraction helper**.
3. Extract PDFs with Docling first (PyPDFLoader fallback).
4. Run `clean_text` → `second_pass_clean` → `final_safe_clean`.
5. Run generic repeated header/footer cleanup and broken/empty-page filtering.
6. Only now run recursive/SLM chunking and the chunking evaluations.
7. Apply the basic structural retrieval-eligibility filter.
8. Select SLM as the final corpus; `final_retrieval_df` is created directly from those final chunks.
9. Run the Qdrant cell: MiniLM embeddings → Qdrant sync. No rebuild occurs here.
10. Use the manual/new-PDF section when testing a new source independently; it uses the same layout-aware + existing-cleaner path.


## Final pipeline audit

**Cleaning now occurs before chunking.** There is no post-chunk corpus rebuild. The only post-chunk filtering is the basic structural retrieval-eligibility check. Scope classification, the binary MiniLM classifier, and NLI are not part of this notebook.
